<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/6_Neo4j_Contexto_Relacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir S6 en Colab"></a>

**Acceso público:** [página del curso](https://jazaineam1.github.io/BigData2026/) · **Laboratorio guiado (checklist paso a paso):** [ábrelo en otra pestaña ↗](https://jazaineam1.github.io/BigData2026/assets/tutoriales/s06-laboratorio-guiado.html)

# Sesión 6 — De la fila priorizada al contexto relacional con Neo4j

## Universidad Central
> ### Facultad de Ingeniería y Ciencias Básicas
> ### Maestría en Analítica de Datos — BIG DATA (64491093)

**Caso conductor:** Compras Claras
**Pregunta profesional:** **Laura ya sabe qué proceso revisar primero. Antes de asignarlo a un auditor, ¿qué relaciones alrededor de ese proceso necesita ver para comprender su contexto?**

**Respuesta corta y herramienta:** Laura necesita seguir una cadena de conexiones —entidad → proceso → proveedor → otro proceso → otra entidad— sin perder el hilo en ningún salto. Eso es una pregunta sobre relaciones que se conectan entre sí, y la herramienta que la responde es una **base de datos de grafos: Neo4j**. El resto de la sesión explica por qué esa herramienta y no otra de las que ya conoces.

### Producto observable

Al terminar tendrás una **ficha relacional de revisión** con:

1. el proceso que llega desde S5;
2. el contexto de prensa heredado;
3. procesos históricos adjudicados de su entidad;
4. proveedores y otras entidades conectadas cuando el dato lo sostenga;
5. el contraste de H2-R, la hipótesis relacional de hoy (pandas ↔ Neo4j);
6. una decisión de modelado y una alternativa descartada;
7. un límite concreto;
8. `s06_contexto_procesos.jsonl`, entrada de la siguiente sesión.

**Entorno:** Google Colab, Python y Cypher; Windows necesita solo navegador.
**Autoevaluación no calificable:** falla aquí, que sale gratis. El hito se valora con la rúbrica del cuaderno.
**Objetivos:** modelar relaciones, verificar una consulta y defender una decisión con evidencia y límites.

## El hilo del evaluador



**Cómo se lee.** Cada sesión entrega el producto que abre la siguiente; aquí reconstruimos solo el contexto necesario para continuar.

**Qué nos dice.** S6 no es un tema nuevo suelto: es la siguiente pregunta sobre el mismo caso.

**Qué NO permite concluir todavía.** Que S6 continúe la cadena no significa que ya sepamos si hay algo irregular — seguimos sin esa evidencia.

**Qué error común.** Tratar cada sesión como un capítulo independiente en vez de un paso de la misma investigación.

## Neo4j: qué es, y por qué aparece aquí

**Neo4j es una base de datos de grafos de propiedades:** representa actores como nodos y conexiones como relaciones dirigidas, ambos con propiedades. Cypher permite expresar el patrón que queremos recorrer.

La necesidad de Laura es seguir Entidad → Proceso → Proveedor → Proceso → Entidad y conservar el contexto de cada conexión.

| Modelo | Cómo representa la pregunta | Qué considerar |
|---|---|---|
| SQL / MongoDB | tablas/documentos y cruces mediante `JOIN` / `$lookup` | también pueden responderla; importan índices, planes y volumen |
| Cassandra | una tabla preparada para una consulta conocida | cambiar la pregunta puede exigir otra tabla o precomputación |
| Neo4j | nodos, relaciones y patrones de caminos | facilita expresar recorridos; el costo crece con las coincidencias y ramificaciones |

**PARA LLEVAR.** Elegimos el grafo por cómo expresa la pregunta. El contrato pandas comprobará la respuesta, no una ventaja de velocidad.

### La misma idea en otros contextos

Estos son ejemplos conceptuales de modelado; no afirmaciones sobre qué motor usa una empresa.

| Contexto | Nodo | Relación | Pregunta |
|---|---|---|---|
| Red profesional | persona, empresa | `TRABAJA_EN`, `CONOCE_A` | ¿qué contacto compartimos? |
| Rutas | intersección | `CONECTA_CON` | ¿qué caminos llegan al destino? |
| Recomendaciones | usuario, contenido | `VIO` | ¿qué contenido comparten usuarios? |
| Transferencias | cuenta | `TRANSFIRIO_A` | ¿qué cuentas están conectadas? |

**Qué error común.** Suponer que una flecha hace constante el costo de cualquier recorrido o demuestra una conducta.

## Mapa de la sesión

| Bloque | Rol | Pregunta | Herramienta | Qué queda |
|---|---|---|---|---|
| 1. Recuperar el ancla | 🧠 ENTIENDE | ¿qué proceso llega desde S5? | Colab + JSON de S5 | proceso + entidad + prensa |
| 2. Preparar contexto | 🧠 ENTIENDE | ¿qué historial rodea esa entidad? | pandas | tabla de contraste |
| 3. Diseñar | 🧠 ENTIENDE | ¿qué es nodo y qué es relación? | papel + cuaderno | Entidad → Proceso → Proveedor |
| 4. Contrato pandas | ▶️ EJECUTA | ¿qué debe responder el grafo? | pandas | resultado esperado |
| 5. AuraDB | ▶️ EJECUTA | ¿cómo levantamos el servicio? | tutorial + Neo4j Aura | conexión real |
| 6. Cypher | ▶️ EJECUTA | ¿cómo cargamos y recorremos relaciones? | Cypher/Neo4j | grafo consultable |
| 7. Verificar | 🧠 + ▶️ | ¿Neo4j conserva la respuesta? | pandas + Neo4j | pandas = Neo4j |
| 8. Hito | ✏️ MODIFICA | ¿qué puede sostener Laura? | Colab | ficha + límite + export |

### Semáforo de código

- 🧠 **ENTIENDE:** debes poder explicarlo con tus palabras.
- ▶️ **EJECUTA:** corre la celda y verifica la salida; **no necesitas escribirla de memoria**.
- ✏️ **MODIFICA:** cambia únicamente el dato señalado y observa qué ocurre.

In [ ]:
#@title Preparar interactividad { display-mode: "form" }
import base64, json, html as html_lib
from IPython.display import display, HTML

def pregunta_codificada(token):
    p = json.loads(base64.b64decode(token).decode("utf-8"))
    uid = f"s06-p{p['numero']}"
    opts = "".join(
        f'<label style="display:block;margin:8px 0"><input type="radio" name="{uid}" value="{i}"> {html_lib.escape(op)}</label>'
        for i, op in enumerate(p["opciones"])
    )
    # Escapar el atributo COMPLETO; la retroalimentación se inserta como texto.
    handler = (
        "const box=this.closest('[data-pregunta]');"
        "const e=box.querySelector('input:checked');"
        "const s=box.querySelector('[aria-live]');"
        "if(!e){s.textContent='Selecciona una opción.';return;}"
        f"const i=Number(e.value),r={json.dumps(p['retro'], ensure_ascii=False)};"
        f"const ok=i==={p['correcta']};"
        "s.textContent=(ok?'Correcto. ':'Incorrecto. ')+r[i];"
        "s.style.background=ok?'#dcfce7':'#fee2e2';"
        "s.style.color=ok?'#14532d':'#7f1d1d';"
        "s.style.padding='12px';"
    )
    handler = html_lib.escape(handler, quote=True)
    contador = str(p['numero']) + (f" de {p['total']}" if p.get('total') else '')
    box = (
        f'<div data-pregunta="{uid}" style="border:2px solid #1e40af;background:#eff6ff;color:#172554;border-radius:12px;padding:15px;margin:14px 0">'
        f'<strong>Pregunta {contador} — {html_lib.escape(p["tema"])}</strong>'
        f'<p style="background:#fef3c7;color:#713f12;padding:10px">{html_lib.escape(p.get("contexto", "Aplica lo que acabas de observar en el caso de Laura."))}</p>'
        f'<p>{html_lib.escape(p["pregunta"])}</p>{opts}'
        f'<button onclick="{handler}" '
        f'style="background:#1e40af;color:white;border:0;border-radius:7px;padding:8px 12px">Verificar respuesta</button>'
        f'<div id="r-{uid}" aria-live="polite"></div></div>'
    )
    display(HTML(box))

def tutorial(url, alto=720):
    box = f'<iframe src="{url}?embed=1" width="100%" height="{alto}" style="border:0;border-radius:10px;background:#faf7ef"></iframe>'
    box += f'<p><a href="{url}" target="_blank">Abrir tutorial en pantalla completa ↗</a></p>'
    display(HTML(box))

print("Soporte S6 listo.")

---
## 1. Recuperar el proceso que Laura abrió en S5

S5 dejó `s05_ancla_s06.json`. Súbelo al panel **Archivos** de Colab y escribe su ruta. Si lo perdiste, la clase no se bloquea: el dataset trae una **ancla pedagógica real** con historial útil.

**OJO.** El respaldo permite aprender Neo4j, pero el hito declara que no se usó el archivo propio.

In [ ]:
#@title Cargar o recuperar el extracto { display-mode: "form" }
import json
import urllib.request
import hashlib
from pathlib import Path
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv"
MANIFEST_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional_manifest.json"
# Se reutiliza el archivo local: también sirve si el docente lo compartió sin red.
rutas = []
for nombre, url in [("s06_contexto_relacional.csv", DATA_URL), ("s06_contexto_relacional_manifest.json", MANIFEST_URL)]:
    archivo = Path(nombre)
    if not archivo.is_file() and (Path("Datos") / nombre).is_file():
        archivo = Path("Datos") / nombre
    if not archivo.is_file():
        try:
            with urllib.request.urlopen(url, timeout=30) as respuesta:
                contenido = respuesta.read()
            archivo.write_bytes(contenido)
        except Exception as exc:
            raise RuntimeError("Carga fallida. Sube el CSV y el manifest del curso a Archivos y repite esta celda.") from exc
    rutas.append(archivo)

datos = pd.read_csv(rutas[0], dtype={"nit_entidad": str, "nit_proveedor": str, "id_proceso": str}, keep_default_na=False)
for columna in ["nit_entidad", "nit_proveedor", "id_proceso"]:
    datos[columna] = datos[columna].str.strip()
for columna in ["precio_base", "valor_adjudicado", "noticias_entidad"]:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")
assert datos["nit_entidad"].ne("").all() and datos["id_proceso"].ne("").all(), "Falta una identidad obligatoria."
manifest = json.loads(rutas[1].read_text(encoding="utf-8-sig"))
huella_datos = hashlib.sha256(rutas[0].read_bytes()).hexdigest()
print("Filas disponibles:", len(datos))
print("Huella SHA256 del extracto:", huella_datos)

In [ ]:
ruta = input("Ruta de s05_ancla_s06.json (Enter = respaldo): ").strip()
if ruta:
    if not Path(ruta).is_file():
        raise FileNotFoundError("No encontré tu archivo. Corrige la ruta o deja Enter para elegir el respaldo explícitamente.")
    ancla_original = json.loads(Path(ruta).read_text(encoding="utf-8-sig"))
    if not isinstance(ancla_original, dict) or not ancla_original.get("nit_entidad") or not ancla_original.get("id_proceso"):
        raise ValueError("El ancla debe contener nit_entidad e id_proceso.")
    par = datos["id_proceso"].eq(str(ancla_original["id_proceso"]).strip()) & datos["nit_entidad"].eq(str(ancla_original["nit_entidad"]).strip()) & datos["tipo_registro"].eq("candidato_s05")
    if not par.any():
        raise ValueError("El proceso y su entidad no corresponden a los candidatos S5 de este extracto. Revisa el archivo.")
    origen_ancla = "archivo propio S5"
else:
    ancla_original = dict(manifest["ancla_pedagogica"])
    origen_ancla = "ancla pedagógica versionada"
print("Origen:", origen_ancla)
print(json.dumps(ancla_original, ensure_ascii=False, indent=2))

### Cómo se lee la entrada

**Cómo se lee.** El ancla identifica un proceso que ya sobrevivió a la regla de S5. El extracto histórico añade hechos adjudicados sin cambiar por qué ese proceso fue priorizado.

**Qué nos dice.** S6 continúa una decisión ya tomada.

**Qué NO permite concluir todavía.** Tener historial contractual no significa que exista una relación problemática.

**Qué error común.** Volver a construir los 77 candidatos. Eso repetiría S5.

### Procedencia y unidad de observación
El extracto se construye con [build_session6_graph_data.py](https://github.com/jazaineam1/BigData2026/blob/main/utils/build_session6_graph_data.py), a partir de los chunks SECOP y la bandeja S5 versionados. El estudiante descarga un archivo del curso; no hace recolección masiva contra SECOP.

**Selección:** entidades mencionadas en el corpus de prensa → contratación directa → cero respuestas (1.000 → 163 → 77). Se toman registros marcados como adjudicados con proveedor informado de esas entidades y se amplía a otras entidades que comparten esos NIT de proveedor. Se deduplican pares proceso–proveedor; se conservan los candidatos como ancla.

**Unidad:** registro candidato o par proceso–proveedor histórico. Una fila no siempre es un contrato único y no es una observación independiente de las demás.

| Campo | Significado y uso |
|---|---|
| `tipo_registro` | candidato S5 o histórico adjudicado; decide si hay relación hacia proveedor |
| `id_proceso`, `nit_entidad`, `nit_proveedor` | identidades reportadas, leídas como texto; no verifican identidad jurídica |
| `nombre_proceso`, `descripcion`, `url_secop` | texto y enlace del proceso, reutilizables en S7 |
| `fecha_publicacion` | fecha del registro; el extracto no impone precedencia respecto del candidato |
| `precio_base`, `valor_adjudicado` | presupuesto base y valor reportado de adjudicación; no son intercambiables |
| `noticias_entidad`, `nivel_menciones` | contexto de prensa de la entidad; no evidencia directa del proceso |
| `es_proceso_candidato_s05`, `es_entidad_candidata_s05` | delimitan la selección S5 y las entidades de referencia |
| `referencia`, `modalidad`, departamentos, nombres | contexto descriptivo del proceso y los actores |

**OJO.** “Histórico” significa adjudicado dentro de este archivo, no necesariamente anterior a tu candidato. No ordenamos alertas temporales ni comparamos riesgo. Un corpus seleccionado por prensa no representa todo SECOP. La fecha de extracción original no está declarada en el manifest; la huella SHA256 identifica el archivo usado, pero no reemplaza esa fecha.

In [ ]:
fechas = pd.to_datetime(datos["fecha_publicacion"], errors="coerce", utc=True, format="mixed")
print("Fechas de publicación válidas:", int(fechas.notna().sum()), "de", len(datos))
print("Intervalo observado:", fechas.min(), "a", fechas.max())

**Cómo se lee.** El intervalo usa las fechas que pudieron interpretarse; no es la fecha de descarga.

**Qué nos dice.** Delimita temporalmente lo que contiene este archivo.

**Qué NO permite concluir todavía.** No establece qué sabía Laura al priorizar: falta una fecha de corte y filtrar cada relación anterior a ella.

**Qué error común.** Convertir relaciones posteriores en señales disponibles antes de la adjudicación. `to_datetime(errors="coerce", utc=True)` convierte fechas y deja inválidas como NaT; no inventa fechas.

## H2-R: la hipótesis relacional de esta sesión

S5 cerró con una hipótesis de prensa: **H1** — ¿aparece literalmente alguno de los 77 IDs de proceso en título o subtítulo de una noticia? El resultado fue `0/77`: **H1 literal refutada**, y la prensa quedó especificada como contexto de entidad, no como evidencia directa de un proceso.

S6 abre una hipótesis distinta, ahora relacional:

> **H2-R.** El proveedor histórico más conectado de la entidad del proceso elegido en S5 está conectado con más entidades que la mediana de esa misma conexión entre los 28 NIT de entidades candidatas de S5 que sí tienen historial.

No es una prueba estadística inferencial: es una comparación empírica y falsable sobre este extracto — el resultado depende de qué proceso elegiste en S5, y varía de una persona a otra.

| Sabemos (llega de S5) | No sabemos todavía |
|---|---|
| proceso, entidad, valor, modalidad | proveedores históricos de esa entidad |
| contexto de prensa (H1 ya refutada) | otras entidades conectadas por el mismo proveedor |
| que el proceso sobrevivió la regla `1.000→163→77` | caminos relacionales entre entidades |

Lo que sigue, con nombres inventados, es exactamente H2-R en miniatura.

---
## 2. El candidato y el historial cumplen funciones distintas

### Ejemplo manual pequeño (nombres inventados, para pensar antes de programar)



Mirando solo este dibujo, sin ninguna tabla: **Constructora Ejemplo S.A.S. aparece conectada con dos entidades distintas** —la Alcaldía de Ejemplo y la Gobernación de Prueba— a través de dos procesos separados. El dibujo hace explícito el camino; una tabla también permite calcularlo mediante cruces. Esto ilustra la conectividad. Para completar H2-R necesitamos una referencia: si los máximos de tres entidades fueran 1, 2 y 4, la mediana sería 2; tener dos conexiones no supera esa mediana.

El proceso candidato que trae S5 puede no estar adjudicado todavía: **no le inventamos un proveedor**. El historial adjudicado —registros adjudicados de la misma entidad— es lo que sí aporta proveedores reales y conexiones observadas.

In [ ]:
#@title Autoevaluación 1 — H2-R en miniatura { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAxLCAidGVtYSI6ICJIMi1SIGVuIG1pbmlhdHVyYSIsICJwcmVndW50YSI6ICLCv1F1w6kgY29tcGFyYWNpw7NuIHB1ZWRlcyBkZWZlbmRlcj8iLCAib3BjaW9uZXMiOiBbIjIgc3VwZXJhIGxhIG1lZGlhbmEgcG9ycXVlIGNvbmVjdGEgbcOhcyBkZSB1bmEgZW50aWRhZC4iLCAiMiBlcyBpZ3VhbCBhIGxhIG1lZGlhbmEgMjsgbm8gbGEgc3VwZXJhLiIsICIyIHN1cGVyYSBsYSBtZWRpYSA3LzMuIiwgIkVsIGRpYnVqbyBkZW11ZXN0cmEgaXJyZWd1bGFyaWRhZC4iXSwgImNvcnJlY3RhIjogMSwgInJldHJvIjogWyJDb25lY3RhciBtw6FzIGRlIHVuYSBlbnRpZGFkIG5vIGJhc3RhOiBIMi1SIGNvbXBhcmEgY29udHJhIGxhIG1lZGlhbmEgZGUgbcOheGltb3MuIEFxdcOtIHZhbGUgMi4iLCAiT3JkZW5hciAxLCAyLCA0IGRlamEgMiBlbiBlbCBjZW50cm8uIERvcyBjb25leGlvbmVzIGlndWFsYW4gZXNlIHVtYnJhbDsgbm8gbG8gc3VwZXJhbi4iLCAiTGEgcmVmZXJlbmNpYSBhY29yZGFkYSBlcyBsYSBtZWRpYW5hLCBubyBsYSBtZWRpYS4gQWRlbcOhcyAyIGVzIG1lbm9yIHF1ZSA3LzMuIiwgIkxhcyBmbGVjaGFzIGRlc2NyaWJlbiBhZGp1ZGljYWNpb25lcyByZWdpc3RyYWRhcy4gRmFsdGFuIGNvbmRpY2lvbmVzIGRlIGNvbXBldGVuY2lhIHkgb3Ryb3MgaGVjaG9zIHBhcmEgZXZhbHVhciBjb25kdWN0YS4iXSwgImNvbnRleHRvIjogIkxvcyBtw6F4aW1vcyBwb3IgZW50aWRhZCBzb24gMSwgMiB5IDQuIEVsIHByb3ZlZWRvciBkZSBFamVtcGxvIGNvbmVjdGEgZG9zIGVudGlkYWRlcy4iLCAidG90YWwiOiAxMH0=")

In [ ]:
#@title Autoevaluación 2 — Evidencia del modelo { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAyLCAidGVtYSI6ICJFdmlkZW5jaWEgZGVsIG1vZGVsbyIsICJwcmVndW50YSI6ICLCv0N1w6FuZG8gZGlidWphciBBREpVRElDQURPX0E/IiwgIm9wY2lvbmVzIjogWyJDdWFuZG8gZWwgcHJvdmVlZG9yIGFwYXJlY2UgZW4gdW5hIG5vdGljaWEgZGUgbGEgZW50aWRhZC4iLCAiQ3VhbmRvIGV4aXN0YSB1biByZWdpc3RybyBoaXN0w7NyaWNvIGFkanVkaWNhZG8gY29uIGlkZW50aWZpY2Fkb3IgZGUgcHJvdmVlZG9yLiIsICJDdWFuZG8gZWwgcHJvY2VzbyB0ZW5nYSB1biBwcmVzdXB1ZXN0byBhbHRvLiIsICJBc2lnbmFuZG8gZWwgcHJvdmVlZG9yIG3DoXMgY29uZWN0YWRvIGRlIGxhIGVudGlkYWQgYWwgY2FuZGlkYXRvLiJdLCAiY29ycmVjdGEiOiAxLCAicmV0cm8iOiBbIkxhIG5vdGljaWEgZXMgY29udGV4dG8gZGUgZW50aWRhZDogbm8gcHJ1ZWJhIGxhIGFkanVkaWNhY2nDs24gZGUgZXNlIHByb2Nlc28uIiwgIkxhIHJlbGFjacOzbiByZXByZXNlbnRhIHVuIGhlY2hvIGRlIGxhIGZ1ZW50ZS4gTGEgY2FyZ2EgZGViZSBleGlnaXIgcmVnaXN0cm8gaGlzdMOzcmljbyB5IE5JVCBpbmZvcm1hZG8uIiwgIkVsIHByZXN1cHVlc3RvIG5vIGlkZW50aWZpY2EgcXVpw6luIHJlY2liacOzIHVuYSBhZGp1ZGljYWNpw7NuLiBObyBwZXJtaXRlIGNvbXBsZXRhciBlc2EgcmVsYWNpw7NuLiIsICJFbCBoaXN0b3JpYWwgcGVydGVuZWNlIGEgb3Ryb3MgcHJvY2Vzb3MuIFRyYXNsYWRhciBzdSBwcm92ZWVkb3IgYWwgY2FuZGlkYXRvIGludmVudGFyw61hIHVuIGhlY2hvLiJdLCAiY29udGV4dG8iOiAiRWwgcHJvY2VzbyBkZSBTNSBlcyBjYW5kaWRhdG87IGVsIGV4dHJhY3RvIGxlIHJlc2VydmEgdW4gcmVnaXN0cm8gc2luIHByb3ZlZWRvci4iLCAidG90YWwiOiAxMH0=")

In [ ]:
nit_deseado = str(ancla_original["nit_entidad"]).strip()
hist = datos[datos["tipo_registro"].eq("historico_adjudicado")].copy()
assert hist["nit_proveedor"].ne("").all(), "Un registro histórico carece de NIT de proveedor."
# La identidad analítica es el NIT reportado. Los nombres se conservan como variantes.
nombres_proveedor = hist.groupby("nit_proveedor")["proveedor"].agg(lambda nombres: " / ".join(sorted(set(nombres))))
variantes = hist.groupby("nit_proveedor")["proveedor"].nunique()
print("NIT de proveedor con varios nombres:", int(variantes.gt(1).sum()))
hist_ancla = hist[hist["nit_entidad"].eq(nit_deseado)]
if hist_ancla.empty:
    print("Tu entidad no tiene historial en el extracto. El trabajo continúa con el respaldo declarado.")
    ancla_trabajo = dict(manifest["ancla_pedagogica"])
    nit_deseado = str(ancla_trabajo["nit_entidad"]).strip()
    hist_ancla = hist[hist["nit_entidad"].eq(nit_deseado)]
    uso_respaldo_s06 = True
else:
    ancla_trabajo = ancla_original
    uso_respaldo_s06 = origen_ancla != "archivo propio S5"
print("Entidad de trabajo:", ancla_trabajo["entidad"])
print("Procesos históricos:", hist_ancla["id_proceso"].nunique())
print("Proveedores distintos:", hist_ancla["nit_proveedor"].nunique())

### Interpretación del contexto histórico

**Cómo se lee.** Los conteos corresponden al historial adjudicado disponible para la entidad de trabajo, no al proceso candidato aislado.

**Qué nos dice.** Hay material relacional suficiente para preguntar por proveedores y conexiones entre procesos.

**Qué NO permite concluir todavía.** Más procesos o proveedores no equivalen a mayor riesgo. Faltan criterios sobre competencia, temporalidad y comportamiento esperado de la entidad.

**Qué error común.** Usar el número de contratos como una puntuación de sospecha.

In [ ]:
#@title Autoevaluación 3 — Identificadores { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAzLCAidGVtYSI6ICJJZGVudGlmaWNhZG9yZXMiLCAicHJlZ3VudGEiOiAiwr9Dw7NtbyBjb25zdHJ1aXIgZWwgY29udHJhdG8gY29tcGFyYWJsZSBjb24gZWwgZ3JhZm8/IiwgIm9wY2lvbmVzIjogWyJBZ3J1cGFyIHBvciBOSVQgeSBjb25zZXJ2YXIgdmFyaWFudGVzIGRlbCBub21icmUuIiwgIkFncnVwYXIgcG9yIG5vbWJyZSB5IE5JVCBwYXJhIGF1bWVudGFyIGxhIHByZWNpc2nDs24uIiwgIkVsaW1pbmFyIHVuYSBhZGp1ZGljYWNpw7NuIHBvcnF1ZSByZXBpdGUgTklULiIsICJDb252ZXJ0aXIgY2FkYSBub21icmUgZW4gdW4gcHJvdmVlZG9yIG51ZXZvLiJdLCAiY29ycmVjdGEiOiAwLCAicmV0cm8iOiBbIkVsIGdyYWZvIGlkZW50aWZpY2EgcG9yIE5JVCByZXBvcnRhZG8uIENvbnNlcnZhbW9zIHZhcmlhbnRlcyBwYXJhIHJldmlzYXIgaWRlbnRpZGFkIHNpbiBzZXBhcmFyIGF1dG9tw6F0aWNhbWVudGUgbG9zIGNvbnRlb3MuIiwgIkHDsWFkaXIgbm9tYnJlIGEgbGEgY2xhdmUgZGl2aWRlIHZhcmlhbnRlcyBkZWwgbWlzbW8gaWRlbnRpZmljYWRvciB5IHB1ZWRlIHJvbXBlciBsYSBpZ3VhbGRhZCBjb24gTmVvNGouIiwgIlVuIHByb3ZlZWRvciBwdWVkZSB0ZW5lciBtdWNob3MgcHJvY2Vzb3MuIEVsaW1pbmFyIHVuYSBhZGp1ZGljYWNpw7NuIHBvciBOSVQgZGVzdHJ1eWUgaGVjaG9zLiIsICJFbCBub21icmUgcHVlZGUgdmFyaWFyIHNpbiBxdWUgY2FtYmllIGVsIGlkZW50aWZpY2Fkb3IuIEhheSBxdWUgcmV2aXNhciBsYXMgdmFyaWFudGVzLCBubyBpbnZlbnRhciBhY3RvcmVzLiJdLCAiY29udGV4dG8iOiAiRG9zIGZpbGFzIHRpZW5lbiBlbCBtaXNtbyBOSVQgZGUgcHJvdmVlZG9yIHkgbm9tYnJlcyBjb24gZXNwYWNpb3MgZGlmZXJlbnRlcy4iLCAidG90YWwiOiAxMH0=")

---
## 3. Diseñar el grafo antes de escribir la consulta final

Antes de escribir Cypher, cinco palabras y nada más.

| Concepto | Qué es | Ejemplo (Constructora Ejemplo S.A.S.) |
|---|---|---|
| Nodo | una entidad del dominio | el proveedor mismo |
| Label | la categoría del nodo | `Proveedor` |
| Propiedad | un dato guardado en el nodo | `nit: "900123456"` |
| Relación | un hecho dirigido entre dos nodos | `ADJUDICADO_A` |
| Camino | una secuencia de nodos y relaciones | Entidad → Proceso → Proveedor |

**Los mismos 5 conceptos, en LinkedIn:** Nodo = una persona · Label = `Persona` o `Empresa` · Propiedad = `nombre: "Ana"` · Relación = `ES_CONTACTO_DE` · Camino = la cadena de contactos que te conecta con alguien que nunca has visto. Es exactamente el mismo vocabulario, sobre datos distintos.

### Cómo se lee `(e:Entidad {nit:"123"})`

- `e` — variable con la que nombras este nodo en el resto de la consulta.
- `Entidad` — el label: la categoría a la que pertenece.
- `{nit:"123"}` — una propiedad que identifica cuál Entidad exactamente.

Con eso ya puedes leer un patrón completo: `(e:Entidad)-[:PUBLICA]->(p:Proceso)` es "un nodo Entidad conectado, mediante la relación PUBLICA, a un nodo Proceso".

## Rúbrica S06

Lee los criterios antes del laboratorio. Total: 100 puntos. **Completo = peso completo; Parcial = mitad; Sin evidencia = 0.** Para convertir a escala 0–5 divide el total entre 20. El respaldo permite entregar un avance; la evidencia de Neo4j se completa después de resolver el acceso.

| Criterio | Completo | Parcial | Sin evidencia | Peso |
|---|---|---|---|---:|
| Continuidad y traza | ID y NIT coherentes, origen/respaldo, autor, fecha y enlace al commit privado | conserva ID y origen pero falta al menos un elemento de traza | no identifica el proceso ni el origen | 15 |
| Modelo | justifica Proceso como nodo para su consulta y explica alternativa descartada | justifica solo una de las dos opciones | reproduce el patrón sin justificación | 20 |
| Ejecución | consulta propia ejecutada en Neo4j, vecindario y filtro de otras entidades correctos | resultados en pandas o solo consulta resuelta en Neo4j | sin resultados ejecutados | 20 |
| Verificación | igualdad con pandas comprobada y carga repetida con conteos iguales | resultados de ambos motores sin comparación o repetición completa | un solo motor o ninguna evidencia | 15 |
| Decisión y H2-R | separa proveedor del máximo, mediana y proveedor explorado; explica elección y entrega JSONL | falta uno o más de esos elementos, pero incluye números propios | respuesta sin números de su ejecución | 15 |
| Límite | conclusión que no puede sostener y dato concreto que falta | nombra solo la conclusión o solo el dato | afirma irregularidad por conectividad | 15 |

Guarda ficha y JSONL en `hitos/s06/` del repositorio **privado** del equipo. Pega el enlace del commit en la entrega del aula; no publiques nombres ni entregas en el repositorio del curso. Conserva también tu consulta propia en ese commit.

### EJERCICIO S06-PATRON — un solo hueco

Completa **solo** el nombre de la relación entre un proceso histórico y el proveedor al que fue adjudicado.

**Qué debe verse si salió bien:** el patrón expresa el hecho contractual y aparece la confirmación “Patrón correcto”.
**Error probable:** dejar `____` o inventar un verbo que no representa el hecho del dato.  
**Qué significa:** el modelo aún no expresa la semántica contractual que luego recorrerá `MATCH`.

<details><summary><strong>Recuperación si te atascaste</strong></summary>
La relación se llama <code>ADJUDICADO_A</code>. Cámbiala y vuelve a ejecutar.
</details>

In [ ]:
RELACION_PROCESO_PROVEEDOR = "____"  # reemplaza únicamente ____
patron_estudiante = f"(p:Proceso)-[:{RELACION_PROCESO_PROVEEDOR}]->(v:Proveedor)"
print(patron_estudiante)

if RELACION_PROCESO_PROVEEDOR != "ADJUDICADO_A":
    raise ValueError("Revisa el hecho contractual que conecta Proceso con Proveedor.")
print("Patrón correcto: la relación expresa una adjudicación observada.")

### Modelo mínimo que usaremos



`(e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)` — así se escribe ese mismo dibujo en Cypher.

| Elemento | Identificador | Decisión |
|---|---|---|
| `Entidad` | NIT | actor que publica |
| `Proceso` | ID SECOP | nodo con texto, valor, modalidad y URL |
| `Proveedor` | NIT | actor adjudicado que puede conectar procesos |
| `PUBLICA` | relación | quién publica el proceso |
| `ADJUDICADO_A` | relación | a quién se adjudicó un proceso histórico |

### La alternativa que descartamos

| Opción | Cómo se vería | Por qué no la usamos hoy |
|---|---|---|
| **Proceso como nodo** (la que usamos) | `(e)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v)` | `Proceso` participa en caminos propios y S7 reutiliza su texto |
| Proceso como propiedad de una relación directa | `(e:Entidad)-[:CONTRATO {id_proceso:"...", valor:...}]->(v:Proveedor)` | más simple, pero un proceso deja de ser algo que puedas recorrer o conectar con otra cosa por sí mismo |

`Proceso` queda como nodo porque hoy participa en caminos y la siguiente sesión reutilizará su texto.

### Función usada: `MERGE`

- **Para qué sirve:** encuentra un nodo o relación que ya existe con esa identidad, o lo crea si no existe usando el patrón indicado. Para garantizar identidad por NIT/ID usamos restricciones únicas.
- **Por qué aparece:** el runtime de Colab se reinicia solo, y el receso pasa a mitad de sesión. Sin `MERGE`, volver a ejecutar la carga crearía un segundo `Proceso 2024-001` idéntico al primero.
- **Intuición en palabras:** es como decir “busca esta persona por su cédula; si no está, regístrala — y protege esa identidad con una restricción única”.
- **Ejemplo manual:** `MERGE (p:Proceso {id:"2024-001"})` la primera vez crea el nodo; ejecutado otra vez, lo encuentra y no crea uno nuevo.
- **Cómo se interpreta la salida:** si el número de nodos no crece al repetir la carga, `MERGE` funcionó como esperado.
- **Error frecuente:** usar `CREATE` en su lugar y terminar con varios nodos duplicados del mismo proceso.

Parámetro del patrón: el `id` estable. Las propiedades que cambian se actualizan con `SET`. Consulta la [documentación de MERGE](https://neo4j.com/docs/cypher-manual/current/clauses/merge/).

### Cypher mínimo

| Construcción | Para qué sirve | Qué devuelve/cambia | Error frecuente |
|---|---|---|---|
| `MERGE` | encuentra o crea un patrón | nodos/relaciones persistidos | creer que siempre crea otro nodo |
| `MATCH` | busca patrones | filas con coincidencias | leerlo como un `SELECT *` sin relaciones |
| `WHERE` | filtra | menos coincidencias | filtrar antes de entender el patrón |
| `WITH` | encadena etapas | variables para la etapa siguiente | olvidar qué variables siguen vivas |
| `RETURN` | define la salida | columnas del resultado | confundir salida con persistencia |
| `ORDER BY` / `LIMIT` | ordena y acota | resultado priorizado | asumir orden si no se pidió |

**PARA LLEVAR.** La flecha es parte de la consulta: no es decoración visual.

In [ ]:
#@title Autoevaluación 4 — Identidad e idempotencia { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA0LCAidGVtYSI6ICJJZGVudGlkYWQgZSBpZGVtcG90ZW5jaWEiLCAicHJlZ3VudGEiOiAiwr9RdcOpIGNvbWJpbmFjacOzbiBwcm90ZWdlIGxhIGlkZW50aWRhZCBkZWwgbm9kbz8iLCAib3BjaW9uZXMiOiBbIkNSRUFURSBlbiBjYWRhIGVqZWN1Y2nDs24uIiwgIk1FUkdFIGluY2x1eWVuZG8gZWwgbm9tYnJlIGNhbWJpYW50ZSBjb21vIGlkZW50aWRhZC4iLCAiTUVSR0UgcG9yIElEIGVzdGFibGUgeSByZXN0cmljY2nDs24gZGUgdW5pY2lkYWQ7IFNFVCBwYXJhIHByb3BpZWRhZGVzLiIsICJMSU1JVCAxIGRlc3B1w6lzIGRlIENSRUFURS4iXSwgImNvcnJlY3RhIjogMiwgInJldHJvIjogWyJDUkVBVEUgYWdyZWdhIG5vZG9zOyByZXBldGlybG8gcHVlZGUgZHVwbGljYXIgZWwgbWlzbW8gcHJvY2Vzby4iLCAiU2kgY2FtYmlhIHVuYSBwcm9waWVkYWQgZGVsIHBhdHLDs24sIE1FUkdFIHB1ZWRlIGJ1c2NhciBvdHJvIHBhdHLDs24uIExhIGlkZW50aWRhZCBkZWJlIHNlciBlbCBJRCBlc3RhYmxlLiIsICJFbCBJRCBkZWZpbmUgZWwgbm9kbywgbGEgcmVzdHJpY2Npw7NuIHByb3RlZ2Ugc3UgdW5pY2lkYWQgeSBTRVQgYWN0dWFsaXphIGF0cmlidXRvcyBzaW4gY2FtYmlhciBzdSBpZGVudGlkYWQuIiwgIkxJTUlUIHJlc3RyaW5nZSBmaWxhcyBkZSBzYWxpZGE7IG5vIGRlc2hhY2UgbG9zIG5vZG9zIGNyZWFkb3MuIl0sICJjb250ZXh0byI6ICJMYSBjYXJnYSBzZSByZXBpdGUgZGVzcHXDqXMgZGVsIHJlY2Vzby4gRWwgSUQgZGVsIHByb2Nlc28gcGVybWFuZWNlIGVzdGFibGUuIiwgInRvdGFsIjogMTB9")

---
## 4. Contrato de resultado: primero pandas

Antes de usar Neo4j calculamos qué proveedores de la entidad ancla también aparecen en otras entidades del extracto. Luego exigiremos a Neo4j la misma respuesta.

In [ ]:
# Ambas métricas usan NIT; el nombre es una etiqueta, no una segunda clave.
prov_ancla = hist_ancla.groupby("nit_proveedor")["id_proceso"].nunique().rename("procesos_con_entidad")
prov_global = hist.groupby("nit_proveedor")["nit_entidad"].nunique().rename("entidades_conectadas")
esperado_pd = pd.concat([prov_ancla, prov_global], axis=1).loc[prov_ancla.index].reset_index()
esperado_pd["procesos_con_entidad"] = esperado_pd["procesos_con_entidad"].astype(int)
esperado_pd["proveedor"] = esperado_pd["nit_proveedor"].map(nombres_proveedor)
esperado_pd = esperado_pd.sort_values(["entidades_conectadas", "procesos_con_entidad", "nit_proveedor"], ascending=[False, False, True]).head(10).reset_index(drop=True)
# Una entidad = un NIT también en el denominador de H2-R.
candidatas_hist = hist[hist["es_entidad_candidata_s05"]].copy()
candidatas_hist["conexiones_proveedor"] = candidatas_hist["nit_proveedor"].map(prov_global)
maximos_candidatas = candidatas_hist.groupby("nit_entidad")["conexiones_proveedor"].max()
MEDIANA_H2R = float(maximos_candidatas.median())
if "mediana_maximo_conectadas_por_nit" in manifest:
    assert MEDIANA_H2R == float(manifest["mediana_maximo_conectadas_por_nit"]), "La referencia por NIT no coincide con el extracto."
proveedor_h2r = esperado_pd.iloc[0].copy()
maximo_h2r = int(proveedor_h2r["entidades_conectadas"])
if uso_respaldo_s06:
    desenlace_h2r_pd = "no evaluable con mi ancla"
elif maximo_h2r > MEDIANA_H2R:
    desenlace_h2r_pd = "conexión más fuerte que la mediana de las candidatas de S5"
else:
    desenlace_h2r_pd = "conexión igual o menor que la mediana de las candidatas de S5"
print("Entidades de referencia:", len(maximos_candidatas))
print("Mediana de referencia (candidatas S5):", MEDIANA_H2R)
print("Proveedor que determina H2-R:", proveedor_h2r["nit_proveedor"], "| máximo:", maximo_h2r)
print("Desenlace H2-R (pandas):", desenlace_h2r_pd)
if uso_respaldo_s06:
    print("Comparación del respaldo:", maximo_h2r, ">", MEDIANA_H2R, "=", maximo_h2r > MEDIANA_H2R)
esperado_pd

### Interpretación del contrato pandas y del desenlace H2-R

**Cómo se lee.** `procesos_con_entidad` cuenta procesos adjudicados de la entidad ancla; `entidades_conectadas` cuenta entidades distintas asociadas al mismo NIT de proveedor. El desenlace H2-R compara tu máximo contra la mediana de esa misma métrica entre los 28 NIT de entidades candidatas de S5 que tienen historial — no contra el universo completo de proveedores, que es demasiado disperso para discriminar.

**Qué nos dice.** El grafo debe reproducir la tabla y el máximo. Con ancla propia se evalúa H2-R; con respaldo se muestra la comparación y se declara su origen. Los 32 pares NIT–nombre del manifest histórico representan 28 NIT; por eso recalculamos una mediana de 21 por identidad, en lugar de 22 por nombre.

**Qué NO permite concluir todavía.** Repetición o conectividad no equivale a favorecimiento, colusión ni irregularidad. Faltarían evidencia sobre competencia, temporalidad, propiedad/representación y criterios de adjudicación.

**Qué error común.** Llamar “sospechoso” al proveedor que queda primero, o llamar “aceptada”/“rechazada” al desenlace de H2-R — es una comparación descriptiva, no una prueba estadística.

**Identidad y denominador:** agrupamos por NIT reportado; conservamos los nombres como variantes. Un identificador compartido por una aseguradora y una unión temporal necesita verificación jurídica antes de interpretar que son el mismo actor. La mediana es de máximos por entidad, no de todos los proveedores.

**Funciones usadas:** `groupby` define la clave; `nunique` cuenta valores distintos; `map` agrega una etiqueta por NIT; `median` devuelve el punto central. No uses el nombre como segunda clave: separaría variantes del mismo identificador.

In [ ]:
#@title Autoevaluación 5 — Mediana de referencia { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA1LCAidGVtYSI6ICJNZWRpYW5hIGRlIHJlZmVyZW5jaWEiLCAicHJlZ3VudGEiOiAiwr9RdcOpIHJlcHJlc2VudGEgbGEgbWVkaWFuYSBIMi1SPyIsICJvcGNpb25lcyI6IFsiTGEgbWVkaWFuYSBkZWwgbsO6bWVybyBkZSBjb250cmF0b3MgZGUgdG9kb3MgbG9zIHByb3ZlZWRvcmVzLiIsICJFbCB1bWJyYWwgcXVlIHBydWViYSByaWVzZ28uIiwgIkVsIHB1bnRvIGNlbnRyYWwgZGUgbG9zIG3DoXhpbW9zIGRlIGNvbmVjdGl2aWRhZCBwb3IgZW50aWRhZCBjYW5kaWRhdGEgY29uIGhpc3RvcmlhbC4iLCAiRWwgcHJvbWVkaW8gZGUgY29uZXhpb25lcyBkZWwgcHJvdmVlZG9yIGVsZWdpZG8gcGFyYSBleHBsb3Jhci4iXSwgImNvcnJlY3RhIjogMiwgInJldHJvIjogWyJIMi1SIGN1ZW50YSBlbnRpZGFkZXMgY29uZWN0YWRhcywgbm8gY29udHJhdG9zLCB5IHJlc3VtZSB1biBtw6F4aW1vIHBvciBlbnRpZGFkIGNhbmRpZGF0YS4iLCAiRXMgdW5hIHJlZmVyZW5jaWEgZGVzY3JpcHRpdmEgZGUgdW5hIG11ZXN0cmEgc2VsZWNjaW9uYWRhOyBubyBleGlzdGUgYXF1w60gdW4gbW9kZWxvIGRlIHJpZXNnby4iLCAiQ2FkYSBOSVQgZGUgZW50aWRhZCBhcG9ydGEgdW4gbcOheGltbzsgb3JkZW5hciBlc29zIG3DoXhpbW9zIHBlcm1pdGUgY2FsY3VsYXIgbGEgbWVkaWFuYSBkZSBjb21wYXJhY2nDs24uIiwgIkVsIHByb3ZlZWRvciBleHBsb3JhZG8gcHVlZGUgbm8gc2VyIGVsIG3DoXhpbW8uIExhIHJlZmVyZW5jaWEgc2UgY29uc3RydXllIGNvbiB0b2RhcyBsYXMgZW50aWRhZGVzIGNhbmRpZGF0YXMgY29uIGhpc3RvcmlhbC4iXSwgImNvbnRleHRvIjogIkVsIGN1YWRlcm5vIGNhbGN1bGEgdW4gbcOheGltbyBwb3IgY2FkYSBlbnRpZGFkIGNhbmRpZGF0YSBxdWUgdGllbmUgaGlzdG9yaWFsLiIsICJ0b3RhbCI6IDEwfQ==")

### RECUPERACIÓN S06 — si Colab reinició
Si perdiste las variables, ejecuta la celda siguiente: reconstruye interactividad, datos, ancla y contrato pandas. Vuelve a indicar el archivo propio o el respaldo. Si todo sigue en memoria, continúa con Aura.

**OJO.** Después debes volver a conectar y repetir la carga; las consultas posteriores necesitan esa conexión. La reconstrucción reutiliza los archivos descargados o subidos. Si el runtime perdió también los archivos y no hay red, sube las copias que entregó el docente.

In [ ]:
#@title Recuperar estado S6 { display-mode: "form" }
# RECUPERACIÓN S06

import base64, json, html as html_lib
from IPython.display import display, HTML

def pregunta_codificada(token):
    p = json.loads(base64.b64decode(token).decode("utf-8"))
    uid = f"s06-p{p['numero']}"
    opts = "".join(
        f'<label style="display:block;margin:8px 0"><input type="radio" name="{uid}" value="{i}"> {html_lib.escape(op)}</label>'
        for i, op in enumerate(p["opciones"])
    )
    # Escapar el atributo COMPLETO; la retroalimentación se inserta como texto.
    handler = (
        "const box=this.closest('[data-pregunta]');"
        "const e=box.querySelector('input:checked');"
        "const s=box.querySelector('[aria-live]');"
        "if(!e){s.textContent='Selecciona una opción.';return;}"
        f"const i=Number(e.value),r={json.dumps(p['retro'], ensure_ascii=False)};"
        f"const ok=i==={p['correcta']};"
        "s.textContent=(ok?'Correcto. ':'Incorrecto. ')+r[i];"
        "s.style.background=ok?'#dcfce7':'#fee2e2';"
        "s.style.color=ok?'#14532d':'#7f1d1d';"
        "s.style.padding='12px';"
    )
    handler = html_lib.escape(handler, quote=True)
    contador = str(p['numero']) + (f" de {p['total']}" if p.get('total') else '')
    box = (
        f'<div data-pregunta="{uid}" style="border:2px solid #1e40af;background:#eff6ff;color:#172554;border-radius:12px;padding:15px;margin:14px 0">'
        f'<strong>Pregunta {contador} — {html_lib.escape(p["tema"])}</strong>'
        f'<p style="background:#fef3c7;color:#713f12;padding:10px">{html_lib.escape(p.get("contexto", "Aplica lo que acabas de observar en el caso de Laura."))}</p>'
        f'<p>{html_lib.escape(p["pregunta"])}</p>{opts}'
        f'<button onclick="{handler}" '
        f'style="background:#1e40af;color:white;border:0;border-radius:7px;padding:8px 12px">Verificar respuesta</button>'
        f'<div id="r-{uid}" aria-live="polite"></div></div>'
    )
    display(HTML(box))

def tutorial(url, alto=720):
    box = f'<iframe src="{url}?embed=1" width="100%" height="{alto}" style="border:0;border-radius:10px;background:#faf7ef"></iframe>'
    box += f'<p><a href="{url}" target="_blank">Abrir tutorial en pantalla completa ↗</a></p>'
    display(HTML(box))

print("Soporte S6 listo.")


import json
import urllib.request
import hashlib
from pathlib import Path
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv"
MANIFEST_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional_manifest.json"
# Se reutiliza el archivo local: también sirve si el docente lo compartió sin red.
rutas = []
for nombre, url in [("s06_contexto_relacional.csv", DATA_URL), ("s06_contexto_relacional_manifest.json", MANIFEST_URL)]:
    archivo = Path(nombre)
    if not archivo.is_file() and (Path("Datos") / nombre).is_file():
        archivo = Path("Datos") / nombre
    if not archivo.is_file():
        try:
            with urllib.request.urlopen(url, timeout=30) as respuesta:
                contenido = respuesta.read()
            archivo.write_bytes(contenido)
        except Exception as exc:
            raise RuntimeError("Carga fallida. Sube el CSV y el manifest del curso a Archivos y repite esta celda.") from exc
    rutas.append(archivo)

datos = pd.read_csv(rutas[0], dtype={"nit_entidad": str, "nit_proveedor": str, "id_proceso": str}, keep_default_na=False)
for columna in ["nit_entidad", "nit_proveedor", "id_proceso"]:
    datos[columna] = datos[columna].str.strip()
for columna in ["precio_base", "valor_adjudicado", "noticias_entidad"]:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")
assert datos["nit_entidad"].ne("").all() and datos["id_proceso"].ne("").all(), "Falta una identidad obligatoria."
manifest = json.loads(rutas[1].read_text(encoding="utf-8-sig"))
huella_datos = hashlib.sha256(rutas[0].read_bytes()).hexdigest()
print("Filas disponibles:", len(datos))
print("Huella SHA256 del extracto:", huella_datos)


ruta = input("Ruta de s05_ancla_s06.json (Enter = respaldo): ").strip()
if ruta:
    if not Path(ruta).is_file():
        raise FileNotFoundError("No encontré tu archivo. Corrige la ruta o deja Enter para elegir el respaldo explícitamente.")
    ancla_original = json.loads(Path(ruta).read_text(encoding="utf-8-sig"))
    if not isinstance(ancla_original, dict) or not ancla_original.get("nit_entidad") or not ancla_original.get("id_proceso"):
        raise ValueError("El ancla debe contener nit_entidad e id_proceso.")
    par = datos["id_proceso"].eq(str(ancla_original["id_proceso"]).strip()) & datos["nit_entidad"].eq(str(ancla_original["nit_entidad"]).strip()) & datos["tipo_registro"].eq("candidato_s05")
    if not par.any():
        raise ValueError("El proceso y su entidad no corresponden a los candidatos S5 de este extracto. Revisa el archivo.")
    origen_ancla = "archivo propio S5"
else:
    ancla_original = dict(manifest["ancla_pedagogica"])
    origen_ancla = "ancla pedagógica versionada"
print("Origen:", origen_ancla)
print(json.dumps(ancla_original, ensure_ascii=False, indent=2))


nit_deseado = str(ancla_original["nit_entidad"]).strip()
hist = datos[datos["tipo_registro"].eq("historico_adjudicado")].copy()
assert hist["nit_proveedor"].ne("").all(), "Un registro histórico carece de NIT de proveedor."
# La identidad analítica es el NIT reportado. Los nombres se conservan como variantes.
nombres_proveedor = hist.groupby("nit_proveedor")["proveedor"].agg(lambda nombres: " / ".join(sorted(set(nombres))))
variantes = hist.groupby("nit_proveedor")["proveedor"].nunique()
print("NIT de proveedor con varios nombres:", int(variantes.gt(1).sum()))
hist_ancla = hist[hist["nit_entidad"].eq(nit_deseado)]
if hist_ancla.empty:
    print("Tu entidad no tiene historial en el extracto. El trabajo continúa con el respaldo declarado.")
    ancla_trabajo = dict(manifest["ancla_pedagogica"])
    nit_deseado = str(ancla_trabajo["nit_entidad"]).strip()
    hist_ancla = hist[hist["nit_entidad"].eq(nit_deseado)]
    uso_respaldo_s06 = True
else:
    ancla_trabajo = ancla_original
    uso_respaldo_s06 = origen_ancla != "archivo propio S5"
print("Entidad de trabajo:", ancla_trabajo["entidad"])
print("Procesos históricos:", hist_ancla["id_proceso"].nunique())
print("Proveedores distintos:", hist_ancla["nit_proveedor"].nunique())


# Ambas métricas usan NIT; el nombre es una etiqueta, no una segunda clave.
prov_ancla = hist_ancla.groupby("nit_proveedor")["id_proceso"].nunique().rename("procesos_con_entidad")
prov_global = hist.groupby("nit_proveedor")["nit_entidad"].nunique().rename("entidades_conectadas")
esperado_pd = pd.concat([prov_ancla, prov_global], axis=1).loc[prov_ancla.index].reset_index()
esperado_pd["procesos_con_entidad"] = esperado_pd["procesos_con_entidad"].astype(int)
esperado_pd["proveedor"] = esperado_pd["nit_proveedor"].map(nombres_proveedor)
esperado_pd = esperado_pd.sort_values(["entidades_conectadas", "procesos_con_entidad", "nit_proveedor"], ascending=[False, False, True]).head(10).reset_index(drop=True)
# Una entidad = un NIT también en el denominador de H2-R.
candidatas_hist = hist[hist["es_entidad_candidata_s05"]].copy()
candidatas_hist["conexiones_proveedor"] = candidatas_hist["nit_proveedor"].map(prov_global)
maximos_candidatas = candidatas_hist.groupby("nit_entidad")["conexiones_proveedor"].max()
MEDIANA_H2R = float(maximos_candidatas.median())
if "mediana_maximo_conectadas_por_nit" in manifest:
    assert MEDIANA_H2R == float(manifest["mediana_maximo_conectadas_por_nit"]), "La referencia por NIT no coincide con el extracto."
proveedor_h2r = esperado_pd.iloc[0].copy()
maximo_h2r = int(proveedor_h2r["entidades_conectadas"])
if uso_respaldo_s06:
    desenlace_h2r_pd = "no evaluable con mi ancla"
elif maximo_h2r > MEDIANA_H2R:
    desenlace_h2r_pd = "conexión más fuerte que la mediana de las candidatas de S5"
else:
    desenlace_h2r_pd = "conexión igual o menor que la mediana de las candidatas de S5"
print("Entidades de referencia:", len(maximos_candidatas))
print("Mediana de referencia (candidatas S5):", MEDIANA_H2R)
print("Proveedor que determina H2-R:", proveedor_h2r["nit_proveedor"], "| máximo:", maximo_h2r)
print("Desenlace H2-R (pandas):", desenlace_h2r_pd)
if uso_respaldo_s06:
    print("Comparación del respaldo:", maximo_h2r, ">", MEDIANA_H2R, "=", maximo_h2r > MEDIANA_H2R)
esperado_pd

print("Estado S6 reconstruido desde archivos versionados o copia local.")

---
## 5. Tutorial visual — AuraDB

**HAZ ESTO AHORA.** Vuelve cuando `RETURN 1 AS conexion` funcione en Query y tengas URI, usuario y contraseña.

El HTML es **instrumental**: muestra el camino de interfaz. Las pantallas dibujadas están rotuladas como representaciones; no se presentan como capturas autenticadas.

Si Aura no está disponible, elige **RESPALDO** en la celda siguiente. Continúa con el mismo proveedor, filtro, ficha y exportación en pandas. La ficha marcará Cypher, CRUD y comparación como pendientes; para completarlos vuelve a conectar y recorre los bloques desde la carga. No se necesita una cuenta de servicio ni tarjeta para el trabajo conceptual.

[Documentación de Aura](https://neo4j.com/docs/aura/) · [Driver de Python](https://neo4j.com/docs/python-manual/current/). Usa una instancia de práctica nueva o dedicada a esta versión del cuaderno: una carga anterior defectuosa puede conservar relaciones ajenas al extracto.

In [ ]:
#@title Abrir tutorial Neo4j Aura { display-mode: "form" }
tutorial('https://jazaineam1.github.io/BigData2026/assets/tutoriales/neo4j-aura-s06-paso-a-paso.html')

In [ ]:
#@title Conectar Aura o activar respaldo { display-mode: "form" }
# Elige una ruta explícita. El respaldo conserva datos y decisiones, pero no ejecuta Cypher.
modo = input("Enter = Aura; escribe RESPALDO si no puedes usar el servicio: ").strip().upper()
if modo not in ["", "RESPALDO"]:
    raise ValueError("Usa Enter o RESPALDO.")
modo_neo4j = modo != "RESPALDO"
if globals().get("driver") is not None:
    driver.close()
driver = None
if modo_neo4j:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neo4j>=6,<7"])
    from getpass import getpass
    from neo4j import GraphDatabase
    URI = input("Connection URI: ").strip()
    USER = input("User name: ").strip()
    PASSWORD = getpass("Password (no se muestra): ")
    if not URI or not USER or not PASSWORD:
        raise ValueError("URI, usuario y contraseña son obligatorios.")
    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    del PASSWORD
    try:
        driver.verify_connectivity()
    except Exception:
        driver.close()
        driver = None
        raise RuntimeError("Conexión fallida. Revisa el diagnóstico del tutorial o repite esta celda y elige RESPALDO.") from None
    print("Conexión Neo4j verificada.")
else:
    print("RESPALDO pandas: Neo4j y CRUD pendientes; no se declarará equivalencia comprobada.")

---
## 6. Identidad y carga idempotente

Primero creamos restricciones. Después `UNWIND` recibe una lista de filas desde Python y `MERGE` reutiliza nodos ya existentes.

**Qué debe verse:** tres restricciones válidas y una carga que puede repetirse sin multiplicar el mismo NIT/ID.  
**Error probable:** autenticación o conectividad antes de ejecutar Cypher. Eso es un problema instrumental, no un problema del modelo; usa el diagnóstico del tutorial.

In [ ]:
if modo_neo4j:
    constraints = [
        "CREATE CONSTRAINT entidad_nit IF NOT EXISTS FOR (e:Entidad) REQUIRE e.nit IS UNIQUE",
        "CREATE CONSTRAINT proceso_id IF NOT EXISTS FOR (p:Proceso) REQUIRE p.id IS UNIQUE",
        "CREATE CONSTRAINT proveedor_nit IF NOT EXISTS FOR (v:Proveedor) REQUIRE v.nit IS UNIQUE",
    ]
    for q in constraints:
        driver.execute_query(q)
    print("Restricciones listas.")
else:
    print("Restricciones Neo4j pendientes (respaldo).")

### Función usada: `UNWIND`

- **Para qué sirve:** convierte una lista (de filas, de diccionarios) en filas individuales que Cypher procesa una por una dentro de la misma consulta.
- **Por qué aparece:** vas a cargar miles de filas de una sola vez desde Python; sin `UNWIND` tendrías que enviar una consulta por fila.
- **Intuición en palabras:** es como decir “toma esta lista de invitados y preséntamelos uno por uno”, para hacer lo mismo con cada uno.
- **Ejemplo manual:** con `filas = [{"id":"P1"}, {"id":"P2"}, {"id":"P3"}]`, `UNWIND $filas AS fila` produce tres filas dentro de una sola consulta: una con `fila.id = "P1"`, otra con `"P2"`, otra con `"P3"`.
- **Cómo se interpreta la salida:** UNWIND produce filas; las cláusulas posteriores determinan si encuentran, crean o filtran elementos.
- **Error frecuente:** usar `filas` (la lista completa) en vez de `fila` (el elemento actual) dentro del patrón.

Parámetro: `$filas`, la lista enviada por Python. Salida: una variable `fila` por elemento. [Referencia UNWIND](https://neo4j.com/docs/cypher-manual/current/clauses/unwind/).

In [ ]:
#@title Autoevaluación 6 — Carga sin relaciones inventadas { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA2LCAidGVtYSI6ICJDYXJnYSBzaW4gcmVsYWNpb25lcyBpbnZlbnRhZGFzIiwgInByZWd1bnRhIjogIsK/UXXDqSBkZWJlIGxsZWdhciBhIGxhIGNhcmdhIEFESlVESUNBRE9fQT8iLCAib3BjaW9uZXMiOiBbIkxhcyAyLjEwOSBmaWxhcyBwb3JxdWUgTmFOIGVxdWl2YWxlIGEgdW4gcHJvdmVlZG9yIGRlc2Nvbm9jaWRvLiIsICJTb2xvIGxhcyAyLjAzMiBoaXN0w7NyaWNhcyBjb24gTklUIGRlIHByb3ZlZWRvciBpbmZvcm1hZG8uIiwgIlNvbG8gbGFzIDc3IGNhbmRpZGF0YXMuIiwgIlVuYSByZWxhY2nDs24gcG9yIGNhZGEgZW50aWRhZCwgc2luIHByb2Nlc28uIl0sICJjb3JyZWN0YSI6IDEsICJyZXRybyI6IFsiVW5hIGF1c2VuY2lhIG5vIGVzIHVuIGFjdG9yLiBDcmVhciB1biBub2RvIGNvbXBhcnRpZG8gcGFyYSBlbGxhIGNvbmVjdGFyw61hIHByb2Nlc29zIHNpbiBldmlkZW5jaWEuIiwgIkVsIHRpcG8gZGUgcmVnaXN0cm8geSBsYSBwcmVzZW5jaWEgZGVsIE5JVCBkZWxpbWl0YW4gbGFzIGFkanVkaWNhY2lvbmVzLiBVTldJTkQgY29udmllcnRlIGVzYSBsaXN0YSBlbiBmaWxhcyBkZW50cm8gZGUgbGEgY29uc3VsdGEuIiwgIkxvcyBjYW5kaWRhdG9zIG5vIGFwb3J0YW4gcHJvdmVlZG9yIGVuIGVzdGUgZXh0cmFjdG87IHN1IGZ1bmNpw7NuIGVzIGFuY2xhciBsYSBwcmVndW50YS4iLCAiTGEgYWRqdWRpY2FjacOzbiBzZSByZWdpc3RyYSBwb3IgcHJvY2VzbyB5IHByb3ZlZWRvci4gU2FsdGFyc2UgZWwgcHJvY2VzbyBwaWVyZGUgbGEgdW5pZGFkIGRlbCBoZWNoby4iXSwgImNvbnRleHRvIjogIkhheSAyLjEwOSBmaWxhczogNzcgY2FuZGlkYXRhcyBzaW4gcHJvdmVlZG9yIHkgMi4wMzIgaGlzdMOzcmljYXMgYWRqdWRpY2FkYXMuIiwgInRvdGFsIjogMTB9")

In [ ]:
cols = [
    "entidad", "nit_entidad", "departamento_entidad", "id_proceso", "referencia",
    "nombre_proceso", "descripcion", "precio_base", "modalidad", "proveedor",
    "nit_proveedor", "departamento_proveedor", "noticias_entidad", "nivel_menciones",
    "tipo_registro", "url_secop", "es_proceso_candidato_s05", "es_entidad_candidata_s05",
]
# Convertir a object permite representar ausencias numéricas como None real.
para_carga = datos[cols].copy()
para_carga["proveedor"] = para_carga["nit_proveedor"].map(nombres_proveedor)
nombres_entidad = datos.groupby("nit_entidad")["entidad"].agg(lambda nombres: " / ".join(sorted(set(nombres))))
para_carga["entidad"] = para_carga["nit_entidad"].map(nombres_entidad)
rows = para_carga.astype(object).where(pd.notna(para_carga), None).to_dict("records")
rows_proveedor = [r for r in rows if r["tipo_registro"] == "historico_adjudicado" and r["nit_proveedor"] not in [None, ""]]
assert len(rows_proveedor) == len(hist), "Las adjudicaciones deben corresponder al historial."
assert all(isinstance(r["nit_proveedor"], str) for r in rows_proveedor)
print("Carga preparada:", len(rows), "filas;", len(rows_proveedor), "adjudicaciones válidas.")

query_base = '''
UNWIND $filas AS fila
MERGE (e:Entidad {nit: toString(fila.nit_entidad)})
SET e.nombre = fila.entidad,
    e.departamento = fila.departamento_entidad,
    e.es_candidata_s05 = fila.es_entidad_candidata_s05,
    e.noticias_entidad = fila.noticias_entidad,
    e.nivel_menciones = fila.nivel_menciones
MERGE (p:Proceso {id: fila.id_proceso})
SET p.referencia = fila.referencia,
    p.nombre = fila.nombre_proceso,
    p.descripcion = fila.descripcion,
    p.valor = fila.precio_base,
    p.modalidad = fila.modalidad,
    p.url = fila.url_secop,
    p.es_candidato_s05 = fila.es_proceso_candidato_s05
MERGE (e)-[:PUBLICA]->(p)
'''
if modo_neo4j:
    driver.execute_query(query_base, filas=rows)

query_proveedor = '''
UNWIND $filas AS fila
MATCH (p:Proceso {id: fila.id_proceso})
MERGE (v:Proveedor {nit: toString(fila.nit_proveedor)})
SET v.nombre = fila.proveedor, v.departamento = fila.departamento_proveedor
MERGE (p)-[:ADJUDICADO_A]->(v)
'''
if modo_neo4j:
    driver.execute_query(query_proveedor, filas=rows_proveedor)
    print("Carga lista:", len(rows), "filas;", len(rows_proveedor), "adjudicaciones.")
else:
    print("Carga Neo4j pendiente (respaldo).")

In [ ]:
if modo_neo4j:
    consulta_tamano = '''
    MATCH (n) WHERE n:Entidad OR n:Proceso OR n:Proveedor
    WITH count(n) AS nodos
    MATCH ()-[r:PUBLICA|ADJUDICADO_A]->()
    RETURN nodos, count(r) AS relaciones
    '''
    antes_carga = driver.execute_query(consulta_tamano).records[0].data()
    driver.execute_query(query_base, filas=rows)
    driver.execute_query(query_proveedor, filas=rows_proveedor)
    despues_carga = driver.execute_query(consulta_tamano).records[0].data()
    carga_repetida = antes_carga == despues_carga
    assert carga_repetida, "La carga repetida alteró los conteos. Revisa IDs y restricciones."
    print("Carga repetida sin crecimiento:", carga_repetida, "|", despues_carga)
else:
    carga_repetida = None
    print("Idempotencia en Neo4j: PENDIENTE.")

**Cómo se lee.** Comparamos nodos y relaciones antes y después de repetir exactamente la carga.

**Qué nos dice.** Si no crecen, la repetición preserva los conteos en esta instancia y ejecución.

**Qué NO permite concluir todavía.** No prueba concurrencia ni ausencia de datos antiguos; hace falta controlar qué había cargado antes.

**Qué error común.** Usar igualdad de conteos como prueba de que cada propiedad es correcta.

**Cómo se lee.** Son 2.109 registros de entrada y 2.032 adjudicaciones históricas; varios registros pueden representar un mismo proceso.

**Qué nos dice.** Los 77 registros candidatos no generan una adjudicación sin evidencia histórica.

**Qué NO permite concluir todavía.** Preparar filas no confirma escritura en Aura; hace falta conexión y consulta al motor.

**Qué error común.** Confundir un NIT ausente con un proveedor, o filas de entrada con nodos creados. Repite la carga y verifica que los conteos del grafo se mantengan.

### Antes de la consulta completa: qué agrega cada salto

Antes de la consulta final, mira qué cambia cuando agregas una flecha más al patrón.

In [ ]:
if modo_neo4j:
    r0 = driver.execute_query("MATCH (e:Entidad) RETURN e.nombre AS entidad LIMIT 5")
    print("0 relaciones -- solo nodos Entidad:")
    print(pd.DataFrame([r.data() for r in r0.records]))

    r1 = driver.execute_query("MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso) RETURN e.nombre AS entidad, p.id AS proceso LIMIT 5")
    print("\n1 relacion (PUBLICA) -- que publico cada entidad:")
    print(pd.DataFrame([r.data() for r in r1.records]))
else:
    print("Demostración Cypher pendiente en Aura (respaldo).")

In [ ]:
if modo_neo4j:
    r2 = driver.execute_query('''
    MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
    RETURN e.nombre AS entidad, p.id AS proceso, v.nombre AS proveedor
    LIMIT 5
    ''')
    print("2 relaciones (PUBLICA + ADJUDICADO_A) -- a quien se adjudico:")
    pd.DataFrame([r.data() for r in r2.records])
else:
    print("Demostración Cypher pendiente en Aura (respaldo).")

| Saltos | Qué responde | Ejemplo de pregunta |
|---|---|---|
| 0 | qué entidades existen | ¿qué entidades cargamos? |
| 1 (`PUBLICA`) | qué publicó cada entidad | ¿qué procesos abrió esta entidad? |
| 2 (`PUBLICA`+`ADJUDICADO_A`) | a quién se le adjudicó lo publicado | ¿a qué proveedor llegó este proceso? |

La consulta completa agrega **dos relaciones más**: Proveedor ← Proceso ← Entidad. El camino entre entidades tiene cuatro relaciones. El conteo incluye la entidad ancla; “otras entidades” requiere excluirla explícitamente.

**Cómo se lee.** Cada fila es una coincidencia del patrón de cero, una o dos relaciones.

**Qué nos dice.** Agregar una relación cambia la unidad de la fila.

**Qué NO permite concluir todavía.** `LIMIT 5` no da una muestra representativa ni un orden; faltan criterios de selección.

**Qué error común.** Contar filas como entidades distintas cuando una entidad publica varios procesos.

In [ ]:
#@title Autoevaluación 7 — WITH y los cuatro saltos { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA3LCAidGVtYSI6ICJXSVRIIHkgbG9zIGN1YXRybyBzYWx0b3MiLCAicHJlZ3VudGEiOiAiwr9Qb3IgcXXDqSBjb25zZXJ2YXIgdiBhbnRlcyBkZWwgc2VndW5kbyBNQVRDSD8iLCAib3BjaW9uZXMiOiBbIlBhcmEgcXVlIGVsIHNpZ3VpZW50ZSBwYXRyw7NuIHVzZSBlbCBtaXNtbyBwcm92ZWVkb3IuIiwgIlBhcmEgcXVlIFdJVEggZ3VhcmRlIHVuYSB0YWJsYSBwZXJtYW5lbnRlLiIsICJQYXJhIHRyYW5zZm9ybWFyIGN1YXRybyByZWxhY2lvbmVzIGVuIHVuYSBzb2xhLiIsICJQYXJhIGV4Y2x1aXIgYXV0b23DoXRpY2FtZW50ZSBsYSBlbnRpZGFkIGFuY2xhLiJdLCAiY29ycmVjdGEiOiAwLCAicmV0cm8iOiBbInYgY29uZWN0YSBsYXMgZG9zIGV0YXBhcy4gUHJpbWVybyBjb250YW1vcyBwcm9jZXNvcyBkZWwgYW5jbGEgeSBkZXNwdcOpcyBidXNjYW1vcyBsYXMgZW50aWRhZGVzIGRlbCBtaXNtbyBwcm92ZWVkb3IuIiwgIldJVEggdHJhbnNtaXRlIHZhcmlhYmxlcyB5IGFncmVnYWRvcyBkZW50cm8gZGUgbGEgY29uc3VsdGE7IG5vIGNyZWEgdW5hIHRhYmxhIHBlcnNpc3RlbnRlLiIsICJFbCBjYW1pbm8gc2lndWUgc2llbmRvIEVudGlkYWTigJNQcm9jZXNv4oCTUHJvdmVlZG9y4oCTUHJvY2Vzb+KAk0VudGlkYWQ6IGN1YXRybyByZWxhY2lvbmVzLiIsICJMYSBleGNsdXNpw7NuIG5lY2VzaXRhIFdIRVJFIG90cmEubml0IDw+ICRhbmNsYS4gV0lUSCBubyBsYSBhZ3JlZ2EuIl0sICJjb250ZXh0byI6ICJMYSBjb25zdWx0YSBjb25zZXJ2YSB2IHkgY291bnQoRElTVElOQ1QgcCkgQVMgcHJvY2Vzb3NfY29uX2VudGlkYWQgbWVkaWFudGUgV0lUSC4iLCAidG90YWwiOiAxMH0=")

---
## 7. La consulta que justifica Neo4j

Ahora recorremos el patrón Entidad → Proceso → Proveedor y, desde ese proveedor, contamos entidades conectadas, incluida la entidad ancla. Más adelante escribirás el filtro para contar solo las otras.

In [ ]:
query_contexto = '''
MATCH (e:Entidad {nit:$nit})-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
WITH v, count(DISTINCT p) AS procesos_con_entidad
MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
RETURN v.nit AS nit_proveedor,
       v.nombre AS proveedor,
       procesos_con_entidad,
       count(DISTINCT otra) AS entidades_conectadas
ORDER BY entidades_conectadas DESC, procesos_con_entidad DESC, nit_proveedor ASC
LIMIT 10
'''
if modo_neo4j:
    neo = driver.execute_query(query_contexto, nit=nit_deseado)
    neo_df = pd.DataFrame([r.data() for r in neo.records], columns=["nit_proveedor", "proveedor", "procesos_con_entidad", "entidades_conectadas"])
    resultado_contexto = neo_df.copy()
else:
    neo_df = None
    resultado_contexto = esperado_pd.copy()
    print("Resultado pandas; consulta Neo4j pendiente.")
resultado_contexto

In [ ]:
coinciden = None
if modo_neo4j:
    cols_cmp = ["nit_proveedor", "procesos_con_entidad", "entidades_conectadas"]
    pd_cmp = esperado_pd[cols_cmp].copy()
    neo_cmp = neo_df[cols_cmp].copy()
    for tabla in [pd_cmp, neo_cmp]:
        tabla["nit_proveedor"] = tabla["nit_proveedor"].astype(str)
        for columna in cols_cmp[1:]:
            tabla[columna] = tabla[columna].astype("int64")
    coinciden = pd_cmp.reset_index(drop=True).equals(neo_cmp.reset_index(drop=True))
    print("pandas == Neo4j:", coinciden)
    assert coinciden, "Revisa NIT, extracto y datos previos en la instancia; no sigas con una comparación distinta."
else:
    print("pandas == Neo4j: PENDIENTE; solo se ejecutó pandas.")

### Interpretación pandas ↔ Neo4j

**Cómo se lee.** Comparamos NIT y las dos métricas en el mismo orden.

**Qué nos dice.** Si aparece True, el grafo reproduce el patrón calculado previamente. En RESPALDO esa equivalencia queda pendiente.

**Qué NO permite concluir todavía.** Es una prueba de corrección, no un benchmark de velocidad ni evidencia de irregularidad.

**Qué error común.** Confundir “la consulta coincide” con “Neo4j es más rápido”.

In [ ]:
#@title Autoevaluación 8 — Corrección y límites { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA4LCAidGVtYSI6ICJDb3JyZWNjacOzbiB5IGzDrW1pdGVzIiwgInByZWd1bnRhIjogIsK/UXXDqSBxdWVkw7MgY29tcHJvYmFkbz8iLCAib3BjaW9uZXMiOiBbIlF1ZSBOZW80aiBlcyBtw6FzIHLDoXBpZG8uIiwgIlF1ZSBsYXMgZW50aWRhZGVzIGNvb3JkaW5hcm9uIGFkanVkaWNhY2lvbmVzLiIsICJRdWUgdG9kbyBTRUNPUCBlc3TDoSByZXByZXNlbnRhZG8uIiwgIlF1ZSBhbWJhcyBpbXBsZW1lbnRhY2lvbmVzIHByb2R1Y2VuIGVzYSByZXNwdWVzdGEgZW4gZWwgZXh0cmFjdG8uIl0sICJjb3JyZWN0YSI6IDMsICJyZXRybyI6IFsiTGEgaWd1YWxkYWQgZGUgcmVzdWx0YWRvcyBubyBtaWRlIGxhdGVuY2lhLiBIYXLDrWFuIGZhbHRhIHRpZW1wb3MgeSBjb25kaWNpb25lcyBjb21wYXJhYmxlcy4iLCAiVW5hIGVzdHJ1Y3R1cmEgY29tcGFydGlkYSBubyBwcnVlYmEgY29vcmRpbmFjacOzbjsgZmFsdGFuIGhlY2hvcyBzb2JyZSBkZWNpc2lvbmVzIHkgdsOtbmN1bG9zLiIsICJFbCBleHRyYWN0byBmdWUgc2VsZWNjaW9uYWRvIHBvciBmaWx0cm9zLiBMYSBjb21wYXJhY2nDs24gbm8gcmVjdXBlcmEgbG9zIHJlZ2lzdHJvcyBleGNsdWlkb3MuIiwgIkNvaW5jaWRlbiBsYXMgY2xhdmVzIHkgbcOpdHJpY2FzIGRlbCBjb250cmF0by4gRXNvIHZlcmlmaWNhIGVzYSBjb25zdWx0YSBzb2JyZSBlc29zIGRhdG9zLCBubyB2ZWxvY2lkYWQgbmkgY29uZHVjdGEuIl0sICJjb250ZXh0byI6ICJFbCByZXN1bHRhZG8gZGUgQXVyYSBjb2luY2lkZSBjb24gcGFuZGFzIGVuIE5JVCB5IGRvcyBtw6l0cmljYXMuIiwgInRvdGFsIjogMTB9")

---
## Demostración guiada — un vecindario rico del extracto

**Demostración guiada; no es tu evidencia individual.** Antes de trabajar con tu propio resultado, vas a ver dibujado — con nodos y flechas de verdad, en Aura — un vecindario con muchos proveedores compartidos: el de la misma entidad que usa el respaldo pedagógico. Sirve para que veas, una vez, en grande, lo que hasta ahora solo viste en tablas.

In [ ]:
nit_ancla_demo = str(manifest["ancla_pedagogica"]["nit_entidad"]).strip()

query_demo_top = '''
MATCH (e:Entidad {nit:$nit})-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
WITH v, count(DISTINCT p) AS procesos_con_entidad
MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
WITH v, procesos_con_entidad, count(DISTINCT otra) AS entidades_conectadas
WHERE entidades_conectadas > 1
RETURN v.nit AS nit_proveedor, v.nombre AS proveedor, procesos_con_entidad, entidades_conectadas
ORDER BY entidades_conectadas DESC, procesos_con_entidad DESC, nit_proveedor ASC
LIMIT 5
'''
if modo_neo4j:
    demo_df = pd.DataFrame([r.data() for r in driver.execute_query(query_demo_top, nit=nit_ancla_demo).records])
else:
    demo_hist = hist[hist["nit_entidad"].eq(nit_ancla_demo)]
    demo_counts = demo_hist.groupby("nit_proveedor")["id_proceso"].nunique().rename("procesos_con_entidad")
    demo_df = pd.concat([demo_counts, prov_global], axis=1).loc[demo_counts.index].reset_index()
    demo_df["proveedor"] = demo_df["nit_proveedor"].map(nombres_proveedor)
    demo_df = demo_df[demo_df["entidades_conectadas"].gt(1)].sort_values(["entidades_conectadas", "procesos_con_entidad", "nit_proveedor"], ascending=[False, False, True]).head(5)
    print("Demostración tabular pandas; dibujo en Aura pendiente.")
demo_df

**Cómo se lee.** Cada fila es un proveedor de la entidad elegida por cantidad de proveedores compartidos, ordenado por cuántas otras entidades también lo adjudicaron.

**Qué nos dice.** Esta demostración usa deliberadamente una entidad con muchos proveedores compartidos — por eso el grafo que verás enseguida es notorio.

**Qué NO permite concluir todavía.** Esta demostración usa la ancla pedagógica, no la tuya. Tu propio resultado (bloque anterior) puede ser más modesto y sigue siendo una respuesta válida a H2-R.

**Qué error común.** Pensar que tu propia ancla “debería” verse igual de conectada que esta demostración.

In [ ]:
top_demo = demo_df.iloc[0]
nit_proveedor_demo = str(top_demo["nit_proveedor"])

query_visual_demo = f'''
MATCH camino_ancla = (e:Entidad {{nit:"{nit_ancla_demo}"}})-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {{nit:"{nit_proveedor_demo}"}})
WITH v, collect(camino_ancla)[0] AS camino_ancla
MATCH camino_otras = (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
WHERE otra.nit <> "{nit_ancla_demo}"
WITH camino_ancla, otra, collect(camino_otras)[0] AS camino_otras
RETURN camino_ancla, camino_otras
LIMIT 8
'''
print(query_visual_demo)

**HAZ ESTO AHORA.** Si estás en Aura, copia la consulta que acabas de imprimir, ve a tu instancia AuraDB → pestaña **Query** (no Colab), pégala y ejecútala ahí. Aura dibuja nodos y flechas automáticamente cuando devuelves caminos (`RETURN camino`).

En RESPALDO usa el esquema conceptual siguiente y conserva la consulta para ejecutarla después. En Aura deberías ver algo con esta forma:



**OJO.** El grafo muestra hasta 8 de las entidades conectadas con este proveedor — se acota para que se pueda leer, no porque las demás no existan.

### Antes de seguir, dos preguntas para el salón

1. En el grafo que acabas de ver, ¿cuál es el nodo puente entre las distintas entidades?
2. ¿Qué representa cada camino que dibujó Aura, y qué NO demuestra por sí solo?

**PARA LLEVAR.** Más conexiones no es lo mismo que una conexión anómala. Un proveedor muy conectado puede operar en un mercado amplio (logística, insumos, papelería…). Para saber si una conexión es inusual haría falta un denominador o un patrón esperado con el que compararla — eso todavía no lo tenemos.

---
## 8. CRUD seguro y tu propio vecindario

Ya viste, en la demostración guiada, un grafo con muchas conexiones. Ahora repites el ejercicio con **tu propio resultado** de la sección 7 — puede ser más modesto, y eso también es una respuesta válida a H2-R.

El CRUD usa `S06-DEMO`; no modificamos un proceso real. Después eliges uno de los **5 proveedores con más entidades conectadas** de tu propio resultado (o todos los disponibles, si tu tabla tiene menos de 5) y abres su vecindario.

**Qué debe verse:** una tabla con entidades y procesos relacionados con el proveedor elegido.
**Error probable:** escoger un número fuera de las opciones mostradas. Significa que tu decisión no corresponde al resultado ejecutado.
**Recuperación:** vuelve a ejecutar y elige un número de la lista; no inventes un NIT.

In [ ]:
if modo_neo4j:
    driver.execute_query('''
    MERGE (e:Entidad {nit:'S06-E'}) SET e.nombre='Entidad demo'
    MERGE (p:Proceso {id:'S06-DEMO'}) SET p.nombre='Proceso demo'
    MERGE (v:Proveedor {nit:'S06-V'}) SET v.nombre='Proveedor demo'
    MERGE (e)-[:PUBLICA]->(p)
    MERGE (p)-[:ADJUDICADO_A]->(v)
    ''')
    r = driver.execute_query("MATCH (p:Proceso {id:'S06-DEMO'}) SET p.estado_revision='revisado' RETURN p.estado_revision AS estado")
    assert r.records[0]["estado"] == "revisado"
    driver.execute_query("MATCH (n) WHERE n.nit IN ['S06-E','S06-V'] OR n.id='S06-DEMO' DETACH DELETE n")
    print("CRUD demo completado y limpiado.")
else:
    print("Demostración Cypher pendiente en Aura (respaldo).")

In [ ]:
if resultado_contexto.empty:
    raise ValueError("No hay proveedores para elegir.")
opciones = resultado_contexto.head(5)
for i, row in opciones.iterrows():
    print(f"{i+1:>2}. {row['proveedor']} | entidades={row['entidades_conectadas']}")
sel = int(input("Número de proveedor: ").strip())
if not 1 <= sel <= len(opciones):
    raise ValueError("Número fuera de rango")
proveedor_elegido = opciones.iloc[sel-1]

if modo_neo4j:
    vec = driver.execute_query('''
    MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {nit:$nit})
    RETURN e.nit AS nit_entidad, e.nombre AS entidad, p.id AS proceso, p.nombre AS nombre_proceso, p.valor AS precio_base
    ORDER BY entidad, valor DESC
    ''', nit=str(proveedor_elegido["nit_proveedor"]))
    vecindario_df = pd.DataFrame([r.data() for r in vec.records])


else:
    vecindario_df = hist[hist["nit_proveedor"].eq(str(proveedor_elegido["nit_proveedor"]))][["nit_entidad", "entidad", "id_proceso", "nombre_proceso", "precio_base"]].drop_duplicates("id_proceso").rename(columns={"id_proceso": "proceso"})
    vecindario_df = vecindario_df.sort_values(["entidad", "precio_base"], ascending=[True, False]).reset_index(drop=True)
    print("Vecindario calculado en pandas; ejecución Neo4j pendiente.")

print("Tamaño observable de tu vecindario:")
print("  Procesos con mi entidad:", int(proveedor_elegido["procesos_con_entidad"]))
print("  Entidades conectadas:", int(proveedor_elegido["entidades_conectadas"]))
print("  Procesos visibles en el vecindario:", len(vecindario_df))
vecindario_df

### Interpretación de tu vecindario

**Cómo se lee.** Cada fila es un proceso conectado al proveedor que elegiste; una misma entidad puede aportar varios procesos. Esta exploración corresponde al proveedor elegido. H2-R se sostiene con el proveedor del máximo, que puede ser otro; la ficha conserva ambos.

**Qué nos dice.** Puedes observar qué entidades y procesos del extracto comparten ese actor contractual y abrir casos concretos para revisión.

**Qué NO permite concluir todavía.** Compartir proveedor no demuestra coordinación, favorecimiento ni irregularidad. Faltan, como mínimo, cronología comparable, condiciones de competencia y vínculos de propiedad/representación cuando la hipótesis los requiera.

**Qué error común.** Convertir el número de conexiones en un “score de riesgo” sin modelo ni denominador.

`precio_base` es el presupuesto base reportado, no el valor adjudicado. Un proceso puede tener varios registros de adjudicación: no sumes ese presupuesto repetidamente como gasto.

### EJERCICIO S06-EXCLUIR — modifica una condición
La columna `entidades_conectadas` incluye tu entidad. Laura quiere saber cuántas **otras** entidades están conectadas al proveedor explorado. Antes de ejecutar, calcula mentalmente: total menos una. Completa el único hueco con el operador Cypher “distinto de”.

**Qué debe verse:** “Otras entidades: N | esperado: N”; N puede ser cero.
**Error probable:** usar igualdad; contarías solo tu entidad. **Recuperación:** consulta el apoyo plegado y repite.

<details><summary>Apoyo si te atascaste</summary>El operador es <code>&lt;&gt;</code>. Si el total es 19, deben quedar 18. El conteo cero también es una respuesta válida.</details>

In [ ]:
OPERADOR_EXCLUSION = "____"  # sustituye por el operador distinto de de Cypher
if OPERADOR_EXCLUSION != "<>":
    raise ValueError("Necesitamos excluir la entidad ancla, no seleccionarla.")
consulta_otras = f'''
MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {{nit:$proveedor}})
WHERE otra.nit {OPERADOR_EXCLUSION} $ancla
RETURN count(DISTINCT otra) AS otras_entidades
'''
if modo_neo4j:
    otras_entidades = int(driver.execute_query(consulta_otras, proveedor=str(proveedor_elegido["nit_proveedor"]), ancla=nit_deseado).records[0]["otras_entidades"])
else:
    otras_entidades = int(vecindario_df.loc[vecindario_df["nit_entidad"].ne(nit_deseado), "nit_entidad"].nunique())
esperadas_otras = int(proveedor_elegido["entidades_conectadas"]) - 1
assert otras_entidades == esperadas_otras
print("Otras entidades:", otras_entidades, "| esperado:", esperadas_otras)
razon_exploracion = input("Con tus números: ¿por qué explorar este proveedor y cuál de la lista descartaste?: ").strip()
if len(razon_exploracion) < 25:
    raise ValueError("Nombra tu elección, otra opción y un dato de tu salida.")

**Cómo se lee.** El total inicial incluía el NIT de la entidad ancla; la nueva consulta lo excluye.

**Qué nos dice.** La resta de una coincide con el filtro por identidad, incluso si el resultado es cero.

**Qué NO permite concluir todavía.** “Otras entidades” no significa competidores; falta conocer mercados y modalidades comparables.

**Qué error común.** Restar un proceso en vez de una entidad distinta.

In [ ]:
#@title Autoevaluación 9 — El proveedor explorado y H2-R { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA5LCAidGVtYSI6ICJFbCBwcm92ZWVkb3IgZXhwbG9yYWRvIHkgSDItUiIsICJwcmVndW50YSI6ICLCv1F1w6kgZGViZSByZWdpc3RyYXIgdHUgZmljaGE/IiwgIm9wY2lvbmVzIjogWyJRdWUgSDItUiB1c2EgMTkgeSBxdWUgaGF5IDE5IG90cmFzIGVudGlkYWRlcy4iLCAiUXVlIGVsIHByb3ZlZWRvciBleHBsb3JhZG8gcHJ1ZWJhIEgyLVIgcG9ycXVlIDE5ID4gMjEuIiwgIk3DoXhpbW8gMzkgcGFyYSBIMi1SOyAxOSBjb25leGlvbmVzIHkgMTggb3RyYXMgZW50aWRhZGVzIHBhcmEgbGEgZXhwbG9yYWNpw7NuLiIsICJTb2xvIGVsIHByb3ZlZWRvciBxdWUgc2UgdmVhIG1lam9yIGVuIGVsIGRpYnVqby4iXSwgImNvcnJlY3RhIjogMiwgInJldHJvIjogWyJFbCBtw6F4aW1vIGRlIGxhIGVudGlkYWQgbm8gY2FtYmlhIGFsIGV4cGxvcmFyIG90cm8gcHJvdmVlZG9yLiBBZGVtw6FzIG90cmFzIGVudGlkYWRlcyBleGNsdXllIGVsIGFuY2xhLiIsICIxOSBubyBzdXBlcmEgMjEgeSBlc2UgcHJvdmVlZG9yIG5vIGRldGVybWluYSBlbCBtw6F4aW1vLiBIYXkgZG9zIHByZWd1bnRhcyB5IGRvcyByZXN1bHRhZG9zLiIsICJTZXBhcmFtb3MgbGEgY29tcGFyYWNpw7NuIGRlIGxhIGVudGlkYWQgZGUgbGEgZGVjaXNpw7NuIGRlIGV4cGxvcmFjacOzbi4gTGEgZmljaGEgY29uc2VydmEgYW1ib3MgTklUIHkgc3VzIG3DqXRyaWNhcy4iLCAiRWwgdGFtYcOxbyBkZWwgZGlidWpvIGRlcGVuZGUgZGVsIGzDrW1pdGUgdmlzdWFsLiBMYSBldmlkZW5jaWEgc29uIGxhcyBjb25zdWx0YXMgeSBzdXMgbsO6bWVyb3MuIl0sICJjb250ZXh0byI6ICJIMi1SIHVzYSBlbCBtw6F4aW1vIDM5IHkgbGEgbWVkaWFuYSAyMS4gRWxlZ2lzdGUgZXhwbG9yYXIgb3RybyBwcm92ZWVkb3IgY29uIDE5IGVudGlkYWRlcywgaW5jbHVpZGEgbGEgYW5jbGEuIiwgInRvdGFsIjogMTB9")

### EJERCICIO S06-CONSULTA — escribe una consulta breve
Construye una consulta que cuente los **procesos distintos de la entidad ancla adjudicados al proveedor elegido**. Usa `$ancla`, `$proveedor` y devuelve una columna `procesos`.

Escribe tu código en la celda siguiente. Puedes apoyarte en el patrón de dos relaciones ya resuelto. **Qué debe verse:** “Consulta propia: N | esperado: N”. **Error probable:** contar filas sin DISTINCT o contar todas las entidades. La comprobación compara con tu propia tabla.

<details><summary>Consulta de recuperación</summary>

```cypher
MATCH (e:Entidad {nit:$ancla})-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {nit:$proveedor})
RETURN count(DISTINCT p) AS procesos
```

En RESPALDO guarda tu consulta para ejecutarla en Aura; el número se comprueba con pandas y se declara pendiente la sintaxis Cypher.
</details>

In [ ]:
# Escribe tu consulta entre las comillas triples; utiliza los dos parámetros indicados.
consulta_propia = '''
'''
if not consulta_propia.strip():
    raise ValueError("Escribe tu consulta; tienes un apoyo plegado encima.")
if modo_neo4j:
    procesos_propios = int(driver.execute_query(consulta_propia, ancla=nit_deseado, proveedor=str(proveedor_elegido["nit_proveedor"])).records[0]["procesos"])
else:
    procesos_propios = int(vecindario_df.loc[vecindario_df["nit_entidad"].eq(nit_deseado), "proceso"].nunique())
    print("Consulta Cypher propia guardada pero pendiente de ejecución en Aura.")
esperados_propios = int(proveedor_elegido["procesos_con_entidad"])
assert procesos_propios == esperados_propios, "Revisa DISTINCT y los dos parámetros."
print("Consulta propia:", procesos_propios, "| esperado:", esperados_propios)

**Cómo se lee.** N es el número de procesos distintos para ese par entidad–proveedor.

**Qué nos dice.** Tu consulta reproduce una métrica del contrato con una selección concreta.

**Qué NO permite concluir todavía.** Ese conteo no mide gasto: falta una medida de adjudicación sin duplicaciones.

**Qué error común.** Confundir consultas que producen el mismo número por casualidad; revisa también el patrón y los parámetros.

### La evidencia no termina en el grafo

Ahora registra dos decisiones que una respuesta genérica no puede inventar por ti:

1. un límite que nombre **qué dato faltaría** antes de una afirmación de riesgo/irregularidad;
2. una alternativa de modelado que descartaste y por qué.

In [ ]:
#@title Autoevaluación 10 — Exportación para S7 { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAxMCwgInRlbWEiOiAiRXhwb3J0YWNpw7NuIHBhcmEgUzciLCAicHJlZ3VudGEiOiAiwr9RdcOpIGhhY2UgbGEgZXhwb3J0YWNpw7NuIGF1ZGl0YWJsZSB5IMO6dGlsIHBhcmEgYsO6c3F1ZWRhIHRleHR1YWw/IiwgIm9wY2lvbmVzIjogWyJHdWFyZGFyIHNvbG8gdW5hIGltYWdlbiBkZWwgZ3JhZm8uIiwgIkd1YXJkYXIgSURzLCBkZXNjcmlwY2lvbmVzLCBVUkwsIHNlbGVjY2nDs24geSBtb3RvciByZWFsIGRlIGVqZWN1Y2nDs24uIiwgIkd1YXJkYXIgbGEgY29udHJhc2XDsWEgZGUgQXVyYSBqdW50byBhbCByZXN1bHRhZG8uIiwgIkd1YXJkYXIgdG9kbyBlbCBkYXRhc2V0IHNpbiBkZWNsYXJhciBlbCBmaWx0cm8uIl0sICJjb3JyZWN0YSI6IDEsICJyZXRybyI6IFsiTGEgaW1hZ2VuIGF5dWRhIGEgbGVlciBjYW1pbm9zLCBwZXJvIG5vIGNvbnNlcnZhIGVsIHRleHRvIGUgSURzIHF1ZSBuZWNlc2l0YSBTNy4iLCAiRWwgdGV4dG8gaGFiaWxpdGEgbGEgYsO6c3F1ZWRhLCBsb3MgSURzIHBlcm1pdGVuIHJhc3RyZWFyIHByb2Nlc29zIHkgbGEgc2VsZWNjacOzbiB5IGVsIG1vdG9yIGRlbGltaXRhbiBsbyBxdWUgZWZlY3RpdmFtZW50ZSBzZSBoaXpvLiIsICJMYXMgY3JlZGVuY2lhbGVzIG5vIHNvbiBldmlkZW5jaWEuIExhIGZpY2hhIGRlYmUgZGVjbGFyYXIgcmVzdWx0YWRvcyB5IGVzdGFkbyBkZWwgbW90b3IsIG51bmNhIGNvbnRyYXNlw7Fhcy4iLCAiU2luIGVsIGNyaXRlcmlvIGRlIHNlbGVjY2nDs24gbm8gc2FiZW1vcyBxdcOpIGNhc29zIHJlcHJlc2VudGEgZWwgYXJjaGl2byBuaSBjw7NtbyBpbnRlcnByZXRhciBsb3MgYXVzZW50ZXMuIl0sICJjb250ZXh0byI6ICJWYXMgYSBjb25zZXJ2YXIgcHJvY2Vzb3MgZGVsIHByb3ZlZWRvciBleHBsb3JhZG8geSBzdXMgZGVzY3JpcGNpb25lcy4iLCAidG90YWwiOiAxMH0=")

In [ ]:
if modo_neo4j:
    proveedor_h2r_neo = neo_df.iloc[0]
    assert str(proveedor_h2r_neo["nit_proveedor"]) == str(proveedor_h2r["nit_proveedor"])
    assert int(proveedor_h2r_neo["entidades_conectadas"]) == maximo_h2r
    if uso_respaldo_s06:
        desenlace_h2r_neo = "no evaluable con mi ancla"
    elif int(proveedor_h2r_neo["entidades_conectadas"]) > MEDIANA_H2R:
        desenlace_h2r_neo = "conexión más fuerte que la mediana de las candidatas de S5"
    else:
        desenlace_h2r_neo = "conexión igual o menor que la mediana de las candidatas de S5"
    assert desenlace_h2r_neo == desenlace_h2r_pd
else:
    desenlace_h2r_neo = "PENDIENTE: no se ejecutó Neo4j"
print("Desenlace H2-R (Neo4j):", desenlace_h2r_neo)
print("Proveedor H2-R:", proveedor_h2r["nit_proveedor"], "| máximo:", maximo_h2r, "| mediana:", MEDIANA_H2R)
print("Proveedor explorado:", proveedor_elegido["nit_proveedor"], "| conexiones:", int(proveedor_elegido["entidades_conectadas"]))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
autor = input("Autor o alias de equipo (guardar solo en repositorio privado): ").strip()
decision_modelo = input("Justifica por qué Proceso debe ser nodo para tu pregunta: ").strip()
if not autor or len(decision_modelo) < 20:
    raise ValueError("Registra autor y justificación de modelado.")
fecha_ejecucion = datetime.now(timezone.utc).isoformat()
limite_estudiante = input("Límite concreto y dato faltante: ").strip()
alternativa_modelo = input("Alternativa de modelado descartada: ").strip()
razon_alternativa = input("¿Por qué la descartaste para esta pregunta?: ").strip()

if len(limite_estudiante) < 25:
    raise ValueError("Nombra la conclusión que no puedes sostener y el dato que falta.")
if len(alternativa_modelo) < 5 or len(razon_alternativa) < 15:
    raise ValueError("Nombra una alternativa real y explica por qué no sirve igual de bien para esta pregunta.")

In [ ]:
#@title Guardar ficha y archivo para S7 { display-mode: "form" }
export = vecindario_df.merge(
    datos[["id_proceso", "descripcion", "modalidad", "url_secop"]].drop_duplicates("id_proceso"),
    left_on="proceso", right_on="id_proceso", how="left", validate="many_to_one"
)
assert export["id_proceso"].notna().all(), "Hay procesos sin correspondencia en el extracto."
export["nit_proveedor_explorado"] = str(proveedor_elegido["nit_proveedor"])
export["motor_ejecucion"] = "Neo4j" if modo_neo4j else "pandas; Neo4j pendiente"
export.to_json("s06_contexto_procesos.jsonl", orient="records", lines=True, force_ascii=False)

hito = f'''# Hito S06 — Ficha relacional de revisión

- Autor: {autor}
- Fecha UTC: {fecha_ejecucion}
- SHA256 del extracto: {huella_datos}
- Carga repetida: {carga_repetida if carga_repetida is not None else "PENDIENTE"}
- Consulta propia: {procesos_propios} procesos; motor {"Neo4j" if modo_neo4j else "pandas; Cypher pendiente"}
- Motor: {"Neo4j" if modo_neo4j else "pandas; Neo4j pendiente"}
- Origen del ancla: {origen_ancla}
- Proceso elegido en S5: {ancla_original.get("id_proceso", "")}
- Proceso/entidad usados para el grafo: {ancla_trabajo.get("id_proceso", "")} — {ancla_trabajo.get("entidad", "")}
- Noticias / nivel: {ancla_trabajo.get("noticias_entidad", "")} / {ancla_trabajo.get("nivel_menciones", "")}
- Respaldo pedagógico: {uso_respaldo_s06}
- H1 (S5): 0/77 → refutada literalmente
- H2-R (S6), desenlace pandas: {desenlace_h2r_pd}
- H2-R (S6), desenlace Neo4j: {desenlace_h2r_neo}
- pandas == Neo4j: {coinciden if coinciden is not None else "PENDIENTE"}
- Proveedor que determina H2-R: {proveedor_h2r["nit_proveedor"]} — {proveedor_h2r["proveedor"]}
- Máximo H2-R: {maximo_h2r}
- Mediana H2-R: {MEDIANA_H2R}
- NIT proveedor explorado: {proveedor_elegido["nit_proveedor"]}
- Otras entidades del proveedor explorado: {otras_entidades}
- Decisión de exploración: {razon_exploracion}
- Proveedor elegido: {proveedor_elegido["proveedor"]}
- Entidades conectadas: {int(proveedor_elegido["entidades_conectadas"])}
- Procesos en el vecindario: {len(vecindario_df)}

## Límite
{limite_estudiante}

## Decisión de modelado
{decision_modelo}

### Alternativa descartada
{alternativa_modelo}

Razón: {razon_alternativa}

## Consulta propia (Cypher)
```cypher
{consulta_propia.strip()}
```
'''
Path("hito_s06_ficha_relacional.md").write_text(hito, encoding="utf-8")
print(hito)

try:
    from google.colab import files
    files.download("hito_s06_ficha_relacional.md")
    files.download("s06_contexto_procesos.jsonl")
except ImportError:
    print("Archivos generados en el runtime.")
print("Exportación S7:", len(export), "procesos; ficha y JSONL guardados.")

### Comprueba tu entrega
La ficha debe mostrar **dos NIT diferenciados** cuando corresponda: proveedor que determina H2-R y proveedor explorado. Verifica máximo, mediana, filtro de otras entidades, decisión propia y estado real del motor. Añade el enlace al commit privado en la entrega del aula.

**Cómo se lee.** El JSONL tiene una fila por proceso del vecindario, con descripción y URL.

**Qué nos dice.** S7 puede buscar texto dentro de la selección que acabas de justificar.

**Qué NO permite concluir todavía.** No es todo SECOP; faltan los procesos excluidos por el recorte y sus fechas comparables.

**Qué error común.** Llamar gasto a `precio_base` o presentar RESPALDO como ejecución en Aura.

---
## Hoja de trucos y puente

```text
UNWIND → convierte una lista en filas
MERGE  → encuentra o crea
MATCH  → busca patrón
WHERE  → filtra
WITH   → encadena
SET    → modifica una propiedad
RETURN → salida
DETACH DELETE → elimina nodo y relaciones
```



**Idea central.** Cassandra organizó datos para una pregunta repetitiva conocida. Neo4j hace de las relaciones una parte explícita de la pregunta.

### Recapitulación
Partimos de un proceso, recuperamos adjudicaciones de su entidad, explicitamos caminos, contrastamos un máximo y elegimos un vecindario. La ficha conserva qué se ejecutó, qué decidimos y qué falta.

**Errores para evitar:** NIT ausente no es proveedor; nombre no es identidad; máximo H2-R no es cualquier proveedor; presupuesto no es gasto; conectividad no es irregularidad.

### Referencias y recursos
- [Fuente y criterios del extracto](https://github.com/jazaineam1/BigData2026/blob/main/utils/build_session6_graph_data.py).
- [MERGE](https://neo4j.com/docs/cypher-manual/current/clauses/merge/) y [UNWIND](https://neo4j.com/docs/cypher-manual/current/clauses/unwind/).
- [Cypher MATCH](https://neo4j.com/docs/cypher-manual/current/clauses/match/) y [WITH](https://neo4j.com/docs/cypher-manual/current/clauses/with/).
- [Driver Python](https://neo4j.com/docs/python-manual/current/) y [Aura](https://neo4j.com/docs/aura/).
- [Tutorial Aura](https://jazaineam1.github.io/BigData2026/assets/tutoriales/neo4j-aura-s06-paso-a-paso.html) y [checklist](https://jazaineam1.github.io/BigData2026/assets/tutoriales/s06-laboratorio-guiado.html).

### Lo que sigue

Laura ya puede ver el vecindario, pero ahora tiene muchos nombres y descripciones de procesos. La nueva pregunta será:

> **¿Cuáles de esos procesos son más relevantes para una búsqueda textual concreta?**

`s06_contexto_procesos.jsonl` será la entrada de Elasticsearch/BM25.

In [ ]:
if driver is not None:
    driver.close()
    print("Conexión Neo4j cerrada.")
else:
    print("Sin conexión Neo4j abierta (respaldo).")